# Build-up pressure analysis: Barça vs Atlético (first half)

Pipeline: tracking tables → possession spells → build-ups → pressure features → statistics → annotated clips → LLM prompts → export.

**Run order:** top to bottom. After a kernel restart, the LLM-prompt section (15) only needs sections 1–9 and section 11 (roster). The video sections (10, 12, 13, 14) can be skipped.

| # | Section | Main output |
|---|---|---|
| 1 | Setup & data loading | `player`, `ball`, pitch constants |
| 2 | Team names | `TEAM_NAMES` |
| 2b/2c | Manual review table & ball-table fixes | `REVIEW`, `ball_raw`, `*_fixed.parquet` |
| 3 | Possession spells | `spells`, `ball.poss_team` |
| 4 | Possession location & thirds | `loc` |
| 5 | Build-up detection | `buildups` |
| 5b/5c | Manual review: inspect, drop, check | `review_log` |
| 5d | Compare with the 2 s snapshot | `merge_gap_comparison.csv` |
| 6 | Pressure measurement | `pf`, `d`, `pr` |
| 7 | Pressure features per build-up | `buildups_p` |
| 8 | Pressed vs not pressed: outcome | landmark table |
| 9 | Loss hazard (2 s) | bootstrap table |
| 10 | Video setup | `VIDEO_PATH`, `STYLE` |
| 11 | Roster & inferred positions | `POS_OF`, `NUM_OF` |
| 12 | Annotation style (v5) | `draw_base_layer` |
| 12b | Carrier check on real frames | `carrier_check/*.jpg` |
| 13 | Camera calibration & ground graphics | homographies, lanes |
| 14 | Final clip rendering | `clips1/` |
| 14b | Dead / untracked check clips | `dead_new/` |
| 15 | LLM prompt for one clip | `prompts/*.txt` |
| 16 | Export for the stats notebook | `pressure_stats_input/` |

*Comments inside cells still use the old cell names (e.g. "Cell 12b" = section 11, "Cell 8" = section 6).*

**Version 4 (new file, v3 kept):** merge gap 5 s; manual carrier segments (`CARRIER_SEGMENTS`, section 2b) applied to the ball table; carrier check on real frames (12b); dead-clip check (14b); comparison with the 2 s snapshot (5d); clips go to `clips3/` and `dead_new/`.

**Pending:** b155 (65 s window to verify), b126/b121 (old Atlético window removed: Barça regained the ball), carrier times for b113, b133, b137, and a manual 'set outcome = lost' action for balls put out by the build-up team's own mistake (b002, b105, b137).

**After changing `REVIEW`:** re-run from section 2b. Build-up numbers can shift, so delete or re-render the clips (section 14 writes to `clips2/`) and re-export (section 16).

## 1. Setup & data loading
Paths, column mapping, loading the player and ball tables, validity filters, and detection of each team's attacking direction.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from scipy.signal import savgol_filter

# ------------------------------------------------------------------ paths
BASE = "C:/Users/user/Desktop/Football_cv_project/football-pipeline (1)/project/barca_atletico_first_half/new_cache"
PLAYER_PATH = BASE + "/player_frame_table_named.parquet"
BALL_PATH = BASE + "/ball_frame_table_named.parquet"

# outputs are written next to the notebook (falls back to the current working directory)
NOTEBOOK_DIR = Path("C:/Users/user/Desktop/Football_cv_project/football-pipeline (1)/project/new/press")
if not NOTEBOOK_DIR.exists():
    NOTEBOOK_DIR = Path.cwd()
OUT_DIR = NOTEBOOK_DIR / "buildup_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------ columns (internal name -> your column name)
PLAYER_COLS = dict(frame_idx="frame_idx", track_id="track_id", team="team", role="role", x="pitch_x", y="pitch_y")
BALL_COLS = dict(frame_idx="frame_idx", carrier_track_id="carrier_track_id", carrier_team="carrier_team",
                 ball_x="ball_pitch_x", ball_y="ball_pitch_y")
PLAYER_OPTIONAL = dict(pitch_valid="pitch_valid", speed_implausible="speed_implausible", x_att="x_att")
BALL_OPTIONAL = dict(possession_team="possession_team", is_stoppage="is_stoppage",
                     homography_valid="homography_valid", ball_speed_implausible="ball_speed_implausible")
OUTFIELD_ROLES = {"player"}

# ------------------------------------------------------------------ pitch and match
FPS = 25
PITCH_LENGTH, PITCH_WIDTH = 105.0, 68.0
TEAM_ATTACKS_POSITIVE_X = {0: True, 1: False}   # placeholder, the loading cell works out the real direction

# ------------------------------------------------------------------ parameters
P = dict(
    # data cleaning
    BOUNDS_TOL_M=3.0,              # player positions further outside the pitch are dropped
    BALL_BOUNDS_TOL_M=5.0,         # same for the ball
    # possession
    POSSESSION_FLICKER_SEC=0.4,    # carrier-team runs shorter than this are absorbed into the previous run
    MIN_HOLD_SEC=1.5,              # a change of possession only counts if the new team holds the ball this long
    # build-up definition (thirds, in the build-up team's own attacking direction)
    THIRD_1_M=PITCH_LENGTH / 3,        # own third ends here (35 m)
    THIRD_2_M=2 * PITCH_LENGTH / 3,    # middle third ends here (70 m); beyond = attacking (final) third
)

# ------------------------------------------------------------------ helpers
F = lambda sec: int(round(sec * FPS))     # seconds -> frames

def bool_runs(mask, frames):
    """Inclusive (start_i, end_i) index pairs of True runs; runs never span a jump in `frames`."""
    mask = np.asarray(mask, bool); frames = np.asarray(frames)
    if len(mask) == 0:
        return []
    brk = np.r_[True, (np.diff(frames) != 1) | (mask[1:] != mask[:-1])]
    starts = np.flatnonzero(brk)
    ends = np.r_[starts[1:] - 1, len(mask) - 1]
    return [(int(s), int(e)) for s, e in zip(starts, ends) if mask[s]]

def mmss(frame):
    s = int(frame / FPS); return f"{s // 60:02d}:{s % 60:02d}"

def to_attack_frame(x, y, team):
    """Pitch coordinates seen from `team`'s own attacking direction (always attacking towards +x)."""
    return (x, y) if TEAM_ATTACKS_POSITIVE_X[int(team)] else (PITCH_LENGTH - x, PITCH_WIDTH - y)

def third_of(x_att):
    """Third of the pitch for a position already in the team's attacking frame."""
    return "own_third" if x_att < P["THIRD_1_M"] else ("middle_third" if x_att < P["THIRD_2_M"] else "attacking_third")

# ------------------------------------------------------------------ check
import sys, scipy
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__)
for name, path in [("player table", PLAYER_PATH), ("ball table", BALL_PATH)]:
    p = Path(path)
    print(f"{name}: {'found' if p.exists() else 'NOT FOUND'}  {p.stat().st_size / 1e6:.1f} MB" if p.exists() else f"{name}: NOT FOUND -> {path}")
print("outputs go to:", OUT_DIR)
print("thirds boundaries (m from own goal):", round(P["THIRD_1_M"], 1), round(P["THIRD_2_M"], 1))

python 3.10.11 | pandas 2.3.3 | numpy 2.2.6 | scipy 1.15.3
player table: found  228.4 MB
ball table: found  3.8 MB
outputs go to: C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\new\press\buildup_output
thirds boundaries (m from own goal): 35.0 70.0


In [2]:
def _read(path, wanted):
    p = Path(path)
    if p.suffix == ".parquet":
        try:
            import pyarrow.parquet as pq
            names = pq.ParquetFile(p).schema.names
            return pd.read_parquet(p, columns=[c for c in wanted if c in names])   # only the columns we need
        except ImportError:
            return pd.read_parquet(p)
    return pd.read_csv(p)

def _prepare(df, cols, optional, name):
    missing = [v for v in cols.values() if v not in df.columns]
    if missing:
        raise KeyError(f"{name}: columns not found {missing}. Edit the column mapping in cell 1.")
    keep = {**cols, **{k: v for k, v in optional.items() if v in df.columns}}
    return df.rename(columns={v: k for k, v in keep.items()})[list(keep)]

def _flag(df, col):
    return df[col].fillna(False).astype(bool)

# ------------------------------------------------------------------ players
player = _prepare(_read(PLAYER_PATH, list(PLAYER_COLS.values()) + list(PLAYER_OPTIONAL.values())),
                  PLAYER_COLS, PLAYER_OPTIONAL, "player table")
n_raw = len(player)
player = player.dropna(subset=["x", "y", "team", "track_id"]).copy()     # also drops referees (no team)
n_pos = len(player)
if "pitch_valid" in player:
    player = player[_flag(player, "pitch_valid")]
if "speed_implausible" in player:
    player = player[~_flag(player, "speed_implausible")]
tol = P["BOUNDS_TOL_M"]
player = player[player.x.between(-tol, PITCH_LENGTH + tol) & player.y.between(-tol, PITCH_WIDTH + tol)].copy()
player["team"] = player["team"].astype(int)
player["track_id"] = player["track_id"].astype(int)
assert set(player["team"].unique()) <= {0, 1}, "team must be 0/1"
print(f"player rows: {n_raw:,} raw -> {n_pos:,} with team and position -> {len(player):,} after validity / speed / bounds filters")

# ------------------------------------------------------------------ attack direction
dir_xatt = {}
if "x_att" in player:
    for t, g in player.dropna(subset=["x_att"]).groupby("team"):
        same = (g["x_att"] - g["x"]).abs().median()
        flipped = (g["x_att"] - (PITCH_LENGTH - g["x"])).abs().median()
        dir_xatt[int(t)] = bool(same <= flipped)
    player = player.drop(columns="x_att")

dir_gk = {}
gk = player[player.role == "goalkeeper"]
if len(gk):
    gk_med = gk.groupby("team")["x"].median()
    print("goalkeeper median x per team:", {int(t): round(float(x), 1) for t, x in gk_med.items()})
    if len(gk_med) == 2:
        dir_gk = {int(t): bool(x < PITCH_LENGTH / 2) for t, x in gk_med.items()}   # own goal at x = 0 -> attacks +x

print(f"attacks towards +x  | config: {TEAM_ATTACKS_POSITIVE_X} | x_att: {dir_xatt} | goalkeepers: {dir_gk}")
if dir_gk and dir_xatt and dir_gk != dir_xatt:
    print("WARNING: x_att and the goalkeeper positions disagree. Using the goalkeepers; check the pipeline's attack_dir / x_att.")
TEAM_ATTACKS_POSITIVE_X = dir_gk if dir_gk else (dir_xatt if dir_xatt else TEAM_ATTACKS_POSITIVE_X)
print("USING team attacks towards +x:", TEAM_ATTACKS_POSITIVE_X)

# ------------------------------------------------------------------ ball
ball = _prepare(_read(BALL_PATH, list(BALL_COLS.values()) + list(BALL_OPTIONAL.values())),
                BALL_COLS, BALL_OPTIONAL, "ball table")
for c in ["carrier_track_id", "carrier_team", "possession_team", "ball_x", "ball_y"]:
    if c in ball:
        ball[c] = ball[c].astype("float64")             # nullable Int -> float with NaN
ok = ball.ball_x.notna() & ball.ball_y.notna()
if "homography_valid" in ball:
    ok &= _flag(ball, "homography_valid")
if "ball_speed_implausible" in ball:
    ok &= ~_flag(ball, "ball_speed_implausible")
tb = P["BALL_BOUNDS_TOL_M"]
ok &= ball.ball_x.between(-tb, PITCH_LENGTH + tb) & ball.ball_y.between(-tb, PITCH_WIDTH + tb)
ball.loc[~ok, ["ball_x", "ball_y"]] = np.nan
if "is_stoppage" in ball:
    ball["is_stoppage"] = _flag(ball, "is_stoppage")
ball = ball.sort_values("frame_idx").drop_duplicates("frame_idx").reset_index(drop=True)

print(f"ball position usable in {ok.mean() * 100:.1f}% of {len(ball):,} frames")
print(f"{player.frame_idx.nunique()} player frames, {player.track_id.nunique()} tracks, {ball.frame_idx.max() / FPS / 60:.1f} min at {FPS} fps")
print("ball columns:", list(ball.columns))

player rows: 1,593,443 raw -> 1,413,951 with team and position -> 1,400,807 after validity / speed / bounds filters
goalkeeper median x per team: {0: 100.9, 1: 8.5}
attacks towards +x  | config: {0: True, 1: False} | x_att: {0: False, 1: True} | goalkeepers: {0: False, 1: True}
USING team attacks towards +x: {0: False, 1: True}
ball position usable in 76.7% of 70,327 frames
70306 player frames, 22 tracks, 46.9 min at 25 fps
ball columns: ['frame_idx', 'carrier_track_id', 'carrier_team', 'ball_x', 'ball_y', 'possession_team', 'is_stoppage', 'homography_valid', 'ball_speed_implausible']


## 2. Team names
Maps team ids (0/1) to club names and checks goalkeepers and player counts.

In [3]:
# --- team names: which team id is which club? ---
pn = pd.read_parquet(PLAYER_PATH, columns=["team", "team_name", "role", "name", "number", "track_id"]).dropna(subset=["team"])
pn["team"] = pn["team"].astype(int)

print("team id vs team_name in the player table (rows):")
print(pn.groupby(["team", "team_name"]).size().to_string())

print("\ngoalkeepers (should be one per team):")
print(pn[pn.role == "goalkeeper"].groupby(["team", "team_name", "name", "number"]).size().to_string())

print("\nplayers per team (distinct names):")
print(pn[pn.role == "player"].groupby(["team", "team_name"])["name"].nunique().to_string())

bn = pd.read_parquet(BALL_PATH, columns=["carrier_team", "carrier_team_name", "possession_team", "possession_team_name"])
print("\nball table: carrier_team vs carrier_team_name")
print(bn.dropna(subset=["carrier_team"]).groupby(["carrier_team", "carrier_team_name"]).size().to_string())
print("\nball table: possession_team vs possession_team_name")
print(bn.dropna(subset=["possession_team"]).groupby(["possession_team", "possession_team_name"]).size().to_string())

# one name per team id, used for labels from now on
TEAM_NAMES = pn.groupby("team")["team_name"].agg(lambda s: s.mode().iloc[0]).to_dict()
print("\nTEAM_NAMES =", TEAM_NAMES)

team id vs team_name in the player table (rows):
team  team_name
0     ATM          716657
1     BAR          697766

goalkeepers (should be one per team):
team  team_name  name   number
0     ATM        Oblak  13        42922
1     BAR        Pena   13        20255

players per team (distinct names):
team  team_name
0     ATM          10
1     BAR          10

ball table: carrier_team vs carrier_team_name
carrier_team  carrier_team_name
0             ATM                  13873
1             BAR                  20449

ball table: possession_team vs possession_team_name
possession_team  possession_team_name
0                ATM                     23475
1                BAR                     32677

TEAM_NAMES = {0: 'ATM', 1: 'BAR'}


### 2b. Manual review table & ball-table corrections
`REVIEW` lists every reviewed clip (identified by team + start time) with its action: `drop`, `fix_possession` (wrong carrier / possession in a time window) or `stoppage` (foul, dead ball). Fixes are applied to the ball table **before** the possession spells are built, so everything downstream recomputes. Fill each `window` from the clock in the clip, then re-run from here.

In [4]:
# =====================================================================================================================
# CELL A  (put it right AFTER section 2 "Team names" and BEFORE section 3 "Possession spells")
# Manual review table + corrections applied to the BALL table before anything is derived from it.
# =====================================================================================================================
import re

# One entry per reviewed clip. Key = clip name WITHOUT ".mp4" (team + start time are what identify it, not the number:
# build-up ids can shift after a fix, so they are never used as the key).
#   action "drop"            : remove the build-up from the analysis (data artefact or not a real build-up)
#   action "fix_possession"  : the carrier / possession label was wrong in the window -> set the true team in that window
#   action "stoppage"        : play was dead (foul, whistle) in the window -> counted as a stoppage, not as a loss
#   window = ("mm:ss", "mm:ss") read from the clock in the clip, inclusive. None = not decided yet (the review cell lists them)
REVIEW = {
    "b060_ATM_16-33": dict(action="drop",           reason="fake pressure (tracking artefact)"),
    "b111_ATM_29-15": dict(action="drop",           reason="fake pressure / not a valid build-up"),
    "b033_ATM_09-36": dict(action="drop",           reason="set piece (corner delivery), not open-play build-up"),
    "b164_ATM_43-22": dict(action="stoppage",       reason="foul, play stopped (not a possession loss)", window=("43:26.9", "43:29.9")),
    "b169_ATM_44-17": dict(action="stoppage",       reason="foul, play stopped (not a possession loss)", window=("44:25.1", "44:28.1")),
    "b114_ATM_30-20": dict(action="fix_possession", reason="fake carrier", window=("30:42.8", "30:44.4")),
    "b126_ATM_34-46": dict(action="fix_possession", reason="NOT a fake carrier: Barça really regained the ball (see CARRIER_SEGMENTS)", window=None),   # old window ("34:50.1","34:54.4") removed
    "b137_ATM_36-54": dict(action="fix_possession", reason="fake carrier", window=("37:02.0", "37:04.4")),
    "b155_BAR_40-37": dict(action="fix_possession", reason="fake carrier", window=None),   # PENDING: suggested window ("40:41.6", "41:46.8") is 65 s, probably a real turnover. Verify on the clip, then set the window or remove the entry.
    "b027_BAR_07-43": dict(action="fix_possession", reason="fake carrier", window=("08:06.2", "08:09.8")),
    "b047_BAR_13-17": dict(action="fix_possession", reason="fake carrier", window=("13:28.0", "13:30.2")),
    "b149_BAR_39-27": dict(action="fix_possession", reason="fake carrier", window=("39:40.8", "39:44.3")),
    "b161_BAR_42-29": dict(action="fix_possession", reason="fake carrier", window=("42:34.4", "42:36.3")),
    "b166_ATM_43-36": dict(action="fix_possession", reason="fake carrier", window=("43:53.8", "43:58.2")),
}
# optional per case: carrier=<track_id> to give the true carrier instead of "unknown" (location then falls back to the ball)

NAME_TO_TEAM = {v: k for k, v in TEAM_NAMES.items()}
_CLIP_RE = re.compile(r"b?(\d+)_([A-Za-z]+)_(\d+)-(\d+)")

def parse_clip(name):
    m = _CLIP_RE.search(Path(name).stem)
    if not m:
        raise ValueError(f"cannot read '{name}' (expected e.g. b114_ATM_30-20)")
    return dict(team_name=m.group(2), team=NAME_TO_TEAM[m.group(2)], start_time=f"{int(m.group(3)):02d}:{int(m.group(4)):02d}")

def to_frame(t):
    """'30:19' or '30:19.5' (or seconds as a number) -> frame index"""
    if isinstance(t, str):
        mm, ss = t.split(":"); t = int(mm) * 60 + float(ss)
    return int(round(float(t) * FPS))

ball_raw = ball.copy()            # untouched copy, used by the review cell below
applied, waiting = [], []
for name, r in REVIEW.items():
    if r["action"] not in ("fix_possession", "stoppage"):
        continue
    if not r.get("window"):
        waiting.append(name); continue
    f0, f1 = to_frame(r["window"][0]), to_frame(r["window"][1])
    m = ball["frame_idx"].between(f0, f1)
    if r["action"] == "fix_possession":
        team = parse_clip(name)["team"]
        ball.loc[m, "possession_team"] = float(team)
        ball.loc[m, "carrier_team"] = float(team)
        ball.loc[m, "carrier_track_id"] = float(r["carrier"]) if r.get("carrier") is not None else np.nan
        ball.loc[m, "is_stoppage"] = False
    else:
        ball.loc[m, "is_stoppage"] = True
    applied.append((name, r["action"], r["window"], int(m.sum())))

print(f"corrections applied: {len(applied)}")
for a in applied:
    print(f"  {a[0]:<16} {a[1]:<15} {a[2][0]} -> {a[2][1]}  ({a[3]} frames)")
if waiting:
    print(f"\nwaiting for a window (not applied yet): {len(waiting)}\n  " + ", ".join(waiting))
    print("  -> run the review cell (CELL B) after section 5 to see what the data says there, then fill 'window' above and re-run from section 3.")

# ---- manual carrier segments (confirmed on the video)
_PN = pn.dropna(subset=["team"]).copy(); _PN["team"] = _PN["team"].astype(int)
TRACK_TEAM = _PN.drop_duplicates("track_id").set_index("track_id")["team"].to_dict()

def by_number(team_name, number):
    """shirt number + team (as printed on the video) -> track_id"""
    g = _PN[(_PN.team_name == team_name) & (_PN.number.astype(float) == float(number))]
    ids = sorted(g.track_id.astype(int).unique())
    if len(ids) != 1:
        raise ValueError(f"{team_name} #{number}: expected exactly one track_id, found {ids}. Use the track_id directly.")
    return ids[0]

CARRIER_SEGMENTS = [
    # b010_ATM_03-22: Barça has the ball, Cubarsi touches at 03:34.4 and 03:38.0
    ("03:34.4", "03:34.9", by_number("BAR", 2)),
    ("03:34.9", "03:38.0", "BAR"),
    ("03:38.0", "03:38.5", by_number("BAR", 2)),
    # b076_BAR_21-20: Casado
    ("21:20.8", "21:22.6", by_number("BAR", 17)),
    # b099_ATM_27-43: long ball, ball still with Atlético
    ("27:53.0", "27:54.6", by_number("ATM", 5)),
    ("27:54.6", "27:56.4", by_number("ATM", 22)),
    ("27:56.4", "27:58.4", "ATM"),                 # ASSUMED end (2 s): adjust when known
    # b105_ATM_28-46
    ("28:51.3", "28:51.8", by_number("ATM", 15)),
    ("28:54.9", "28:55.4", by_number("BAR", 23)),  # assumed throw-in
    # pending (no times yet): b113, b121, b133, b137
]
def apply_carrier_segments(df, full_table=False):
    """Works on the in-memory `ball` (full_table=False) or on the full parquet table (full_table=True: also clears the stale carrier name / number columns).
    Third item of a segment: a track_id (that player carries the ball) or a team name such as "ATM" (that team has it, carrier unknown)."""
    for seg in CARRIER_SEGMENTS:
        t0, t1, who = seg[0], seg[1], seg[2]
        if isinstance(who, str):
            team, tid = NAME_TO_TEAM[who], None
        else:
            tid = int(who); team = TRACK_TEAM[tid]
        m = df["frame_idx"].between(to_frame(t0), to_frame(t1))
        na_ = lambda c: pd.NA if str(df[c].dtype).startswith("Int") else np.nan
        cast = lambda c, v: int(v) if str(df[c].dtype).startswith("Int") else float(v)
        df.loc[m, "carrier_track_id"] = na_("carrier_track_id") if tid is None else cast("carrier_track_id", tid)
        df.loc[m, "carrier_team"] = cast("carrier_team", team)
        df.loc[m, "possession_team"] = cast("possession_team", team)
        if "is_stoppage" in df:
            df.loc[m, "is_stoppage"] = False
        if full_table:
            for c in ("carrier_team_name", "possession_team_name"):
                if c in df:
                    df.loc[m, c] = TEAM_NAMES[team]
            for c in df.columns:
                if c.startswith("carrier") and c not in ("carrier_track_id", "carrier_team", "carrier_team_name"):
                    df.loc[m, c] = na_(c)
    return df

ball = apply_carrier_segments(ball)
print(f"carrier segments applied: {len(CARRIER_SEGMENTS)}")

# =====================================================================================================================


corrections applied: 9
  b164_ATM_43-22   stoppage        43:26.9 -> 43:29.9  (77 frames)
  b169_ATM_44-17   stoppage        44:25.1 -> 44:28.1  (75 frames)
  b114_ATM_30-20   fix_possession  30:42.8 -> 30:44.4  (41 frames)
  b137_ATM_36-54   fix_possession  37:02.0 -> 37:04.4  (61 frames)
  b027_BAR_07-43   fix_possession  08:06.2 -> 08:09.8  (91 frames)
  b047_BAR_13-17   fix_possession  13:28.0 -> 13:30.2  (56 frames)
  b149_BAR_39-27   fix_possession  39:40.8 -> 39:44.3  (89 frames)
  b161_BAR_42-29   fix_possession  42:34.4 -> 42:36.3  (49 frames)
  b166_ATM_43-36   fix_possession  43:53.8 -> 43:58.2  (111 frames)

waiting for a window (not applied yet): 2
  b126_ATM_34-46, b155_BAR_40-37
  -> run the review cell (CELL B) after section 5 to see what the data says there, then fill 'window' above and re-run from section 3.
carrier segments applied: 9


### 2c. (Optional) Save the corrected ball table as a new file
Writes `*_fixed.parquet` next to the original; the original is never overwritten.

In [5]:
# ================================================================== OPTIONAL: save the corrected ball table as a NEW file
# The original parquet is never written to. Set SAVE_FIXED_BALL_TABLE = False to skip. To use the fixed table in other notebooks,
# point BALL_PATH (section 1) at FIXED_BALL_PATH.
SAVE_FIXED_BALL_TABLE = True
if SAVE_FIXED_BALL_TABLE:
    tbl = pd.read_parquet(BALL_PATH)
    na = lambda c: pd.NA if str(tbl[c].dtype).startswith("Int") else np.nan
    for name, r in REVIEW.items():
        if r["action"] not in ("fix_possession", "stoppage") or not r.get("window"):
            continue
        f0, f1 = to_frame(r["window"][0]), to_frame(r["window"][1])
        m = tbl["frame_idx"].between(f0, f1)
        if r["action"] == "fix_possession":
            t = parse_clip(name)["team"]
            for c in ("carrier_team", "possession_team"):
                tbl.loc[m, c] = t
            for c in ("carrier_team_name", "possession_team_name"):
                if c in tbl: tbl.loc[m, c] = TEAM_NAMES[t]
            tbl.loc[m, "carrier_track_id"] = r.get("carrier", na("carrier_track_id"))
            for c in tbl.columns:      # other per-carrier columns are no longer valid unless a true carrier was given
                if c.startswith("carrier") and c not in ("carrier_track_id", "carrier_team", "carrier_team_name") and r.get("carrier") is None:
                    tbl.loc[m, c] = na(c)
        else:
            tbl.loc[m, "is_stoppage"] = True
    tbl = apply_carrier_segments(tbl, full_table=True)      # manual carrier segments, written into the copy as well
    FIXED_BALL_PATH = str(Path(BALL_PATH).with_name(Path(BALL_PATH).stem + "_fixed.parquet"))
    assert FIXED_BALL_PATH != str(BALL_PATH), "refusing to overwrite the original table"
    tbl.to_parquet(FIXED_BALL_PATH, index=False)
    print("saved", FIXED_BALL_PATH, "| original untouched:", BALL_PATH)


saved C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\new_cache\ball_frame_table_named_fixed.parquet | original untouched: C:/Users/user/Desktop/Football_cv_project/football-pipeline (1)/project/barca_atletico_first_half/new_cache/ball_frame_table_named.parquet


## 3. Possession spells
Stable-hold rule on the raw possession label (short flickers absorbed). A spell is an unbroken stretch of one team, tagged as turnover or restart.

In [6]:
P["TURNOVER_MAX_GAP_SEC"] = 2.0   # team change with at most this long a gap = turnover; a longer gap (ball out / stoppage) = restart

def absorb_short_runs(vals, min_len):
    """Runs shorter than min_len take the label of the run before them."""
    vals = vals.copy(); n = len(vals)
    brk = np.r_[True, vals[1:] != vals[:-1]]
    starts = np.flatnonzero(brk); ends = np.r_[starts[1:], n]
    prev = None
    for s, e in zip(starts, ends):
        if prev is not None and (e - s) < min_len:
            vals[s:e] = prev
        else:
            prev = vals[s]
    return vals

fr = ball["frame_idx"].to_numpy()
raw = ball["possession_team"].to_numpy(float).copy()
raw[ball["is_stoppage"].to_numpy()] = np.nan             # dead time never belongs to a possession

# stable-hold rule, applied separately inside each contiguous stretch (never across a stoppage or gap)
stable = raw.copy()
min_len = F(P["MIN_HOLD_SEC"])
for s, e in bool_runs(~np.isnan(raw), fr):
    stable[s:e + 1] = absorb_short_runs(raw[s:e + 1], min_len)
ball["poss_team"] = stable

def n_changes(arr):
    total = 0
    for s, e in bool_runs(~np.isnan(arr), fr):
        seg = arr[s:e + 1]; total += int((seg[1:] != seg[:-1]).sum())
    return total

# spells = unbroken stretches of one possession team
valid = ~np.isnan(stable)
brk = np.r_[True, (np.diff(fr) != 1) | (stable[1:] != stable[:-1]) | (valid[1:] != valid[:-1])]
ball["spell_id"] = np.where(valid, np.cumsum(brk), -1)

spells = (ball[valid].groupby("spell_id")
          .agg(team=("poss_team", "first"), start_frame=("frame_idx", "first"),
               end_frame=("frame_idx", "last"), n_frames=("frame_idx", "count"))
          .reset_index())
spells["team"] = spells["team"].astype(int)
spells["duration_sec"] = spells["n_frames"] / FPS
prev_end, prev_team = spells["end_frame"].shift(), spells["team"].shift()
spells["gap_sec"] = (spells["start_frame"] - prev_end - 1) / FPS
spells["start_type"] = np.where(prev_team.notna() & (prev_team != spells["team"]) &
                                (spells["gap_sec"] <= P["TURNOVER_MAX_GAP_SEC"]), "turnover", "restart")

# ------------------------------------------------------------------ checks
print(f"possession known in {valid.mean() * 100:.1f}% of {len(ball):,} frames "
      f"(stoppage frames excluded: {ball['is_stoppage'].mean() * 100:.1f}%)")
print(f"possession changes: {n_changes(ball['possession_team'].to_numpy(float).copy())} raw -> "
      f"{n_changes(stable)} after the {P['MIN_HOLD_SEC']} s stable-hold rule")

summ = (spells.groupby("team")
        .agg(spells=("spell_id", "size"), minutes=("duration_sec", lambda s: s.sum() / 60),
             median_sec=("duration_sec", "median"), mean_sec=("duration_sec", "mean"), max_sec=("duration_sec", "max")))
summ.index = [TEAM_NAMES[t] for t in summ.index]
print("\npossession spells by team:")
print(summ.round(2))

print("\nhow spells start (turnover = other team had the ball just before; restart = after a stoppage / gap):")
print(spells.groupby(["team", "start_type"]).size().unstack(fill_value=0).rename(index=TEAM_NAMES))

print("\nspell duration quantiles (s):", spells["duration_sec"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_dict())
spells.head(8)

possession known in 77.2% of 70,327 frames (stoppage frames excluded: 15.7%)
possession changes: 149 raw -> 96 after the 1.5 s stable-hold rule

possession spells by team:
     spells  minutes  median_sec  mean_sec  max_sec
ATM     111    14.96        5.24      8.09    41.28
BAR     111    21.23        6.04     11.48    55.72

how spells start (turnover = other team had the ball just before; restart = after a stoppage / gap):
start_type  restart  turnover
team                         
ATM              52        59
BAR              52        59

spell duration quantiles (s): {0.1: 1.8, 0.25: 3.4, 0.5: 5.5, 0.75: 12.5, 0.9: 23.7}


,spell_id,team,start_frame,end_frame,n_frames,duration_sec,gap_sec,start_type
0,13,0,12,299,288,11.52,NaN,restart
1,14,1,300,795,496,19.84,0.00,turnover
2,62,1,843,1586,744,29.76,1.88,restart
3,83,1,1607,1655,49,1.96,0.80,restart
4,84,0,1656,1881,226,9.04,0.00,turnover
5,345,0,2142,2168,27,1.08,10.40,restart
6,346,1,2169,2735,567,22.68,0.00,turnover
7,347,0,2736,2998,263,10.52,0.00,turnover


### 3b. Merge same-team gaps
Spells of the same team separated by a short gap with no stoppage are merged.

In [13]:
P["MERGE_GAP_SEC"] = 2.0

spells_raw = spells.copy()
sp = spells.reset_index(drop=True).copy()
row_of = lambda f: np.searchsorted(fr, f)                 # frame_idx -> row in `ball`

stop = ball["is_stoppage"].to_numpy().astype(int)
cs = np.r_[0, np.cumsum(stop)]
prev_end = sp["end_frame"].shift()
sp["gap_frames"] = sp["start_frame"] - prev_end - 1
sp["gap_stop_frames"] = [cs[row_of(s)] - cs[row_of(pe) + 1] if pd.notna(pe) else np.nan
                         for s, pe in zip(sp["start_frame"], prev_end)]

same_team = sp["team"].eq(sp["team"].shift())
mergeable = same_team & (sp["gap_frames"] <= F(P["MERGE_GAP_SEC"])) & (sp["gap_stop_frames"] == 0)
sp["group"] = (~mergeable).cumsum()

spells = (sp.groupby("group")
          .agg(team=("team", "first"), start_frame=("start_frame", "first"), end_frame=("end_frame", "last"),
               start_type=("start_type", "first"), gap_sec=("gap_sec", "first"), n_parts=("spell_id", "size"))
          .reset_index(drop=True))
spells.insert(0, "spell_id", np.arange(len(spells)))
spells["n_frames"] = spells["end_frame"] - spells["start_frame"] + 1
spells["duration_sec"] = spells["n_frames"] / FPS

# frame-level table follows the merged spells (gap frames inside a merged spell now carry its team)
ball["spell_id"] = -1
for r in spells.itertuples():
    a, b_ = row_of(r.start_frame), row_of(r.end_frame)
    ball.loc[a:b_, "spell_id"] = r.spell_id
    ball.loc[a:b_, "poss_team"] = r.team

# ------------------------------------------------------------------ checks
print(f"spells: {len(spells_raw)} -> {len(spells)} after merging {int(mergeable.sum())} same-team gaps "
      f"(<= {P['MERGE_GAP_SEC']} s, no stoppage inside)")

g = sp.iloc[1:].copy()
g["same_team"] = same_team.iloc[1:].to_numpy()
g["has_stoppage"] = g["gap_stop_frames"] > 0
print("\ngaps between consecutive spells (count):")
print(pd.crosstab(g["same_team"].map({True: "same team", False: "team changes"}),
                  g["has_stoppage"].map({True: "contains stoppage", False: "no stoppage"})))
print("\ngap length quantiles (s), by whether the team changes:")
for flag, name in [(True, "same team"), (False, "team changes")]:
    q = (g[g["same_team"] == flag]["gap_frames"] / FPS).quantile([0.25, 0.5, 0.75, 0.9]).round(2).to_dict()
    print(f"  {name}: {q}")

summ = (spells.groupby("team").agg(spells=("spell_id", "size"), minutes=("duration_sec", lambda s: s.sum() / 60),
                                   median_sec=("duration_sec", "median"), max_sec=("duration_sec", "max")))
summ.index = [TEAM_NAMES[t] for t in summ.index]
print("\nspells after merging:"); print(summ.round(2))
print("\nspells shorter than 2 s:", int((spells["duration_sec"] < 2).sum()), "of", len(spells))
print("start types:", spells["start_type"].value_counts().to_dict())

spells: 167 -> 167 after merging 0 same-team gaps (<= 2.0 s, no stoppage inside)

gaps between consecutive spells (count):
has_stoppage  contains stoppage  no stoppage
same_team                                   
same team                    10            6
team changes                 16          134

gap length quantiles (s), by whether the team changes:
  same team: {0.25: 5.69, 0.5: 11.54, 0.75: 15.02, 0.9: 17.98}
  team changes: {0.25: 0.0, 0.5: 0.0, 0.75: 1.22, 0.9: 4.94}

spells after merging:
     spells  minutes  median_sec  max_sec
ATM      86    15.72        6.80    65.24
BAR      81    22.04        7.68    95.28

spells shorter than 2 s: 16 of 167
start types: {'turnover': 118, 'restart': 49}


In [21]:
print(len(sp), "raw spells |", int(same_team.sum()), "consecutive same-team pairs")
print((sp.loc[same_team, "gap_frames"] / FPS).describe().round(1).to_string())
print("pairs with a stoppage inside the gap:", int((sp.loc[same_team, "gap_stop_frames"] > 0).sum()))

167 raw spells | 16 consecutive same-team pairs
count    16.0
mean     11.1
std       5.9
min       3.0
25%       5.7
50%      11.5
75%      15.0
max      23.4
pairs with a stoppage inside the gap: 10


## 4. Possession location & thirds
Where the possessing team is on the pitch (carrier position, else ball, with short interpolation), in the team's own attacking frame.

In [14]:
P["LOC_FILL_FRAMES"] = F(0.5)      # interpolate position dropouts up to this long
P["LOC_MEDIAN_FRAMES"] = 11        # ~0.44 s median filter inside a spell (stops thirds flickering on jitter)

car = player[["frame_idx", "track_id", "x", "y"]].rename(columns={"track_id": "carrier_track_id", "x": "cx", "y": "cy"})
car["carrier_track_id"] = car["carrier_track_id"].astype(float)
loc = (ball[["frame_idx", "carrier_track_id", "ball_x", "ball_y", "poss_team", "spell_id"]]
       .merge(car, on=["frame_idx", "carrier_track_id"], how="left"))

loc["src"] = np.where(loc.cx.notna(), "carrier", np.where(loc.ball_x.notna(), "ball", "none"))
x = loc["cx"].fillna(loc["ball_x"]); y = loc["cy"].fillna(loc["ball_y"])
lim = P["LOC_FILL_FRAMES"]
x_i = x.interpolate(limit=lim, limit_area="inside"); y_i = y.interpolate(limit=lim, limit_area="inside")
loc.loc[x.isna() & x_i.notna(), "src"] = "interpolated"
loc["x"], loc["y"] = x_i, y_i

in_spell = loc["spell_id"] >= 0
loc.loc[~in_spell, ["x", "y"]] = np.nan                       # only possession frames carry a location

k = P["LOC_MEDIAN_FRAMES"]
for c in ["x", "y"]:
    loc[c] = loc.groupby("spell_id")[c].transform(lambda s: s.rolling(k, center=True, min_periods=1).median())
loc.loc[~in_spell, ["x", "y"]] = np.nan

# attacking frame of the team that has the ball
attacks_pos = loc["poss_team"].map(TEAM_ATTACKS_POSITIVE_X).fillna(True).astype(bool).to_numpy()
loc["x_att"] = np.where(attacks_pos, loc["x"], PITCH_LENGTH - loc["x"])
loc["y_att"] = np.where(attacks_pos, loc["y"], PITCH_WIDTH - loc["y"])
loc["third"] = pd.cut(loc["x_att"], [-np.inf, P["THIRD_1_M"], P["THIRD_2_M"], np.inf],
                      labels=["own_third", "middle_third", "attacking_third"], right=False)
loc.loc[~in_spell, "third"] = np.nan

# ------------------------------------------------------------------ checks
poss_frames = loc[in_spell]
print(f"possession frames: {len(poss_frames):,}")
print("\nwhere the location comes from (% of possession frames):")
print((poss_frames["src"].value_counts(normalize=True) * 100).round(1).to_string())
print(f"no usable location: {poss_frames['x_att'].isna().mean() * 100:.1f}%")

print("\nshare of possession time in each third (%), in the possessing team's own attacking direction:")
t3 = (poss_frames.dropna(subset=["x_att"]).groupby(["poss_team", "third"], observed=False).size()
      .unstack(fill_value=0))
t3 = t3.div(t3.sum(axis=1), axis=0) * 100
t3.index = [TEAM_NAMES[int(t)] for t in t3.index]
print(t3.round(1))

print("\nx_att (m from own goal) when in possession:")
print(poss_frames.dropna(subset=["x_att"]).groupby("poss_team")["x_att"].describe().round(1)
      .rename(index=TEAM_NAMES).to_string())
print("\nx range raw:", round(float(poss_frames["x"].min()), 1), "to", round(float(poss_frames["x"].max()), 1),
      "| y range:", round(float(poss_frames["y"].min()), 1), "to", round(float(poss_frames["y"].max()), 1))

possession frames: 56,627

where the location comes from (% of possession frames):
src
carrier         59.5
ball            30.4
none             7.1
interpolated     2.9
no usable location: 5.3%

share of possession time in each third (%), in the possessing team's own attacking direction:
third  own_third  middle_third  attacking_third
ATM         48.3          36.4             15.2
BAR         19.3          47.5             33.3

x_att (m from own goal) when in possession:
             count  mean   std  min   25%   50%   75%    max
poss_team                                                   
ATM        22099.0  39.5  26.5 -3.1  18.1  35.9  55.8  109.5
BAR        31526.0  58.9  24.6 -4.9  41.4  61.5  75.3  110.0

x range raw: -4.9 to 110.0 | y range: -4.5 to 72.9


C:\Users\user\AppData\Local\Temp\ipykernel_13744\2321402398.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  attacks_pos = loc["poss_team"].map(TEAM_ATTACKS_POSITIVE_X).fillna(True).astype(bool).to_numpy()


## 5. Build-up detection
A build-up ends as `success` (ball holds beyond the entry line), `lost` (turnover to the opponent) or `dead_or_untracked`. Includes a sensitivity check on the entry line.

In [15]:
P["ENTRY_HOLD_SEC"] = 0.5      # ball must stay beyond the entry line this long to count as a success
P["RESET_BUFFER_M"] = 10.0     # after a success, the ball must drop this far below the line before a NEW build-up starts

spell_start_type = spells.set_index("spell_id")["start_type"].to_dict()
spell_pos = {sid: i for i, sid in enumerate(spells.spell_id)}
lp = loc[loc.spell_id >= 0][["frame_idx", "spell_id", "poss_team", "x_att", "y_att"]]
spell_groups = {sid: g for sid, g in lp.groupby("spell_id", sort=True)}

def cut_buildups(t_entry=None, reset_buffer=None, entry_hold_sec=None):
    t_entry = P["THIRD_2_M"] if t_entry is None else t_entry
    reset_below = t_entry - (P["RESET_BUFFER_M"] if reset_buffer is None else reset_buffer)
    entry_f = F(P["ENTRY_HOLD_SEC"] if entry_hold_sec is None else entry_hold_sec)
    rows, n_attack_spells = [], 0
    for sid, g in spell_groups.items():
        frames = g["frame_idx"].to_numpy()
        xa, ya = g["x_att"].to_numpy(float), g["y_att"].to_numpy(float)
        team = int(g["poss_team"].iloc[0]); known = ~np.isnan(xa)
        if not known.any():
            continue
        entries = [(s, e) for s, e in bool_runs(known & (xa >= t_entry), frames) if e - s + 1 >= entry_f]
        resets = [(s, e) for s, e in bool_runs(known & (xa < reset_below), frames) if e - s + 1 >= entry_f]
        first_known = int(np.flatnonzero(known)[0])
        if xa[first_known] < t_entry:
            start_i, reason = first_known, spell_start_type[sid]
        else:
            n_attack_spells += 1                              # spell began inside the attacking zone (high regain / restart)
            nxt = [s for s, e in resets if s > first_known]   # a build-up starts only after a real drop back
            if not nxt:
                continue
            start_i, reason = nxt[0], "recycled_after_attack"

        while True:
            hit = [(s, e) for s, e in entries if s > start_i]
            if hit:
                s, e = hit[0]
                outcome, detail = "success", "entered_attacking_third"
                end_frame, last_i, end_x, end_y = frames[s], e, xa[s], ya[s]
            else:
                last_i = len(frames) - 1
                kn = np.flatnonzero(known)[-1]
                end_frame, end_x, end_y = frames[-1], xa[kn], ya[kn]
                nxt_sp = spells.iloc[spell_pos[sid] + 1] if spell_pos[sid] + 1 < len(spells) else None
                if nxt_sp is None:
                    outcome, detail = "data_end", "end_of_data"
                elif nxt_sp.team != team and nxt_sp.start_type == "turnover":
                    outcome, detail = "lost", "turnover_to_opponent"
                elif nxt_sp.team != team:
                    outcome, detail = "dead_or_untracked", "out_or_stoppage_then_opponent"
                else:
                    outcome, detail = "dead_or_untracked", "same_team_after_gap"
            rows.append(dict(
                spell_id=sid, team=team, team_name=TEAM_NAMES[team],
                start_frame=int(frames[start_i]), end_frame=int(end_frame), start_time=mmss(frames[start_i]),
                duration_sec=(int(end_frame) - int(frames[start_i]) + 1) / FPS,
                start_x_att=xa[start_i], start_y_att=ya[start_i], start_third=third_of(xa[start_i]),
                start_reason=reason, outcome=outcome, end_detail=detail,
                end_x_att=end_x, end_y_att=end_y, max_x_att=float(np.nanmax(xa[start_i:last_i + 1])),
            ))
            if outcome != "success":
                break
            nxt = [s for s, e in resets if s > last_i]        # after a success: new build-up only after a real drop back
            if not nxt:
                break
            start_i, reason = nxt[0], "recycled_after_attack"
    out = pd.DataFrame(rows).sort_values("start_frame").reset_index(drop=True)
    return out, n_attack_spells

buildups, n_attack_spells = cut_buildups()
buildups.insert(0, "buildup_id", np.arange(len(buildups)))

# ------------------------------------------------------------------ checks
print(f"{len(buildups)} build-ups from {len(spells)} possession spells "
      f"({n_attack_spells} spells began inside the attacking zone)")
oc = buildups.groupby(["team_name", "outcome"]).size().unstack(fill_value=0)
print("\noutcome by team:"); print(oc.to_string())
dec = oc.reindex(columns=["success", "lost"], fill_value=0)
print("\nsuccess rate among decided build-ups, %:"); print((dec["success"] / dec.sum(axis=1) * 100).round(1).to_string())
print("share of ALL build-ups that were successes, %:"); print((oc["success"] / oc.sum(axis=1) * 100).round(1).to_string())
print("\nstart third by team:"); print(buildups.groupby(["team_name", "start_third"]).size().unstack(fill_value=0).to_string())
print("\nstart reason by team:"); print(buildups.groupby(["team_name", "start_reason"]).size().unstack(fill_value=0).to_string())
print("\nend detail:"); print(buildups.groupby(["outcome", "end_detail"]).size().to_string())

depth = buildups.loc[buildups.outcome == "success", "max_x_att"] - P["THIRD_2_M"]
print("\nhow far past the 70 m line successes reached (m): ", depth.quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_dict())
print(f"successes that reached less than 2 m past the line: {(depth < 2).mean() * 100:.0f}%")
print(f"build-ups shorter than 1 s: {(buildups.duration_sec < 1).sum()} of {len(buildups)}")

# sensitivity to the position of the entry line
rs = []
for t in [66, 68, 70, 72, 75]:
    b, _ = cut_buildups(t_entry=t)
    for name, gg in b.groupby("team_name"):
        rs.append(dict(entry_line_m=t, team=name, buildups=len(gg), success=int((gg.outcome == "success").sum()),
                       lost=int((gg.outcome == "lost").sum()), undecided=int((gg.outcome == "dead_or_untracked").sum())))
sens = pd.DataFrame(rs)
sens["success_of_decided_%"] = (sens.success / (sens.success + sens.lost) * 100).round(1)
print("\nsensitivity to the entry line:"); print(sens.to_string(index=False))
print()
print(buildups.head(8)[["buildup_id", "team_name", "start_time", "duration_sec", "start_third", "start_reason", "outcome", "max_x_att"]].round(1).to_string())

158 build-ups from 167 possession spells (33 spells began inside the attacking zone)

outcome by team:
outcome    dead_or_untracked  lost  success
team_name                                  
ATM                       18    46       14
BAR                       12    21       47

success rate among decided build-ups, %:
team_name
ATM    23.3
BAR    69.1
share of ALL build-ups that were successes, %:
team_name
ATM    17.9
BAR    58.8

start third by team:
start_third  middle_third  own_third
team_name                           
ATM                    25         53
BAR                    59         21

start reason by team:
start_reason  recycled_after_attack  restart  turnover
team_name                                             
ATM                               2       20        56
BAR                              22       14        44

end detail:
outcome            end_detail                   
dead_or_untracked  out_or_stoppage_then_opponent    17
                   same_team_after

### 5b. Manual review: inspect, drop and log
For cases still waiting for a `window`, prints what the ball table says around the end of the build-up. Drops the build-ups marked `drop` (build-up numbers are kept) and prints the review log.

In [16]:
# =====================================================================================================================
# CELL B  (put it right AFTER section 5 "Build-up detection")
# 1) shows, for every case still waiting for a window, what the ball table says around the end of the build-up
# 2) drops the build-ups marked "drop" and keeps a log of every exclusion
# =====================================================================================================================
_pn = pn.drop_duplicates("track_id").set_index("track_id")
def _who(tid):
    if pd.isna(tid):
        return "-"
    t = int(tid); r = _pn.loc[t] if t in _pn.index else None
    return f"#{int(r.number) if r is not None and pd.notna(r.number) else '?'} {r['name'] if r is not None else '?'} ({TEAM_NAMES.get(int(_pn.loc[t].team) if r is not None and 'team' in _pn.columns and pd.notna(_pn.loc[t].team) else -1, '?')})"

def _clock(frame):
    s = frame / FPS; return f"{int(s // 60):02d}:{s % 60:04.1f}"

def show_case(name, pre=4.0, post=6.0, step=0.5):
    info = parse_clip(name)
    row = buildups[(buildups.team_name == info["team_name"]) & (buildups.start_time == info["start_time"])]
    if row.empty:
        print(f"{name}: no build-up with team {info['team_name']} starting at {info['start_time']} (ids/times may have changed)"); return
    b = row.iloc[0]
    print(f"\n{name}: build-up {int(b.buildup_id)}, {b.team_name}, {b.start_time} -> {_clock(b.end_frame)}, outcome {b.outcome} ({b.end_detail})")
    print(f"{'clock':>8}  {'carrier (ball table)':<34} poss_team  stoppage   ball x,y")
    a, z = int(b.end_frame - pre * FPS), int(b.end_frame + post * FPS)
    w = ball_raw[(ball_raw.frame_idx >= a) & (ball_raw.frame_idx <= z)].iloc[::max(1, int(step * FPS))]
    for r in w.itertuples():
        pt = "-" if pd.isna(r.possession_team) else TEAM_NAMES[int(r.possession_team)]
        bx = "-" if pd.isna(r.ball_x) else f"{r.ball_x:.0f},{r.ball_y:.0f}"
        mark = "  <-- build-up ends" if abs(r.frame_idx - b.end_frame) < step * FPS / 2 else ""
        print(f"{_clock(r.frame_idx):>8}  {_who(r.carrier_track_id):<34} {pt:<9}  {str(bool(r.is_stoppage)):<8}   {bx}{mark}")

for name in waiting:
    show_case(name)

# ---- drop the build-ups marked "drop" (identified by team + start time)
log = []
drop_mask = pd.Series(False, index=buildups.index)
for name, r in REVIEW.items():
    info = parse_clip(name)
    hit = (buildups.team_name == info["team_name"]) & (buildups.start_time == info["start_time"])
    if r["action"] == "drop":
        drop_mask |= hit
    log.append(dict(clip=name, action=r["action"], reason=r["reason"], found=int(hit.sum()), window=r.get("window")))
review_log = pd.DataFrame(log)
n_before = len(buildups)
buildups = buildups[~drop_mask].copy()                       # buildup_id is kept as it is (clip numbers stay valid)
print(f"\ndropped {int(drop_mask.sum())} build-ups ({n_before} -> {len(buildups)}); fixes/stoppages are applied upstream")
print(review_log.to_string(index=False))
print("\noutcome by team after review:")
print(buildups.groupby(["team_name", "outcome"]).size().unstack(fill_value=0).to_string())



b126_ATM_34-46: build-up 112, ATM, 34:46 -> 34:50.0, outcome lost (turnover_to_opponent)
   clock  carrier (ball table)               poss_team  stoppage   ball x,y
 34:46.0  -                                  BAR        False      70,36
 34:46.5  -                                  BAR        False      76,34
 34:47.0  #15 Lenglet (ATM)                  ATM        False      78,34
 34:47.5  #15 Lenglet (ATM)                  ATM        False      78,33
 34:48.0  #15 Lenglet (ATM)                  ATM        False      81,29
 34:48.4  #15 Lenglet (ATM)                  ATM        False      82,26
 34:48.9  #2 Gimenez (ATM)                   ATM        False      83,26
 34:49.4  #2 Gimenez (ATM)                   ATM        False      84,23
 34:49.9  #2 Gimenez (ATM)                   ATM        False      85,17  <-- build-up ends
 34:50.4  #6 Gavi (BAR)                      BAR        False      85,17
 34:50.8  #6 Gavi (BAR)                      BAR        False      87,15
 34:51.3  #6

### 5c. Check the manual fixes
Fixed carriers must no longer end as `lost / turnover_to_opponent`; fouls must end as `dead_or_untracked`.

In [17]:
# ================================================================== CHECK: the manual fixes did what they should
# fixed carriers must no longer end as lost / turnover_to_opponent; fouls must be dead_or_untracked
for name, r in REVIEW.items():
    if r["action"] not in ("fix_possession", "stoppage"):
        continue
    i = parse_clip(name)
    hit = buildups[(buildups.team_name == i["team_name"]) & (buildups.start_time == i["start_time"])]
    state = hit[["buildup_id", "outcome", "end_detail"]].to_string(header=False, index=False) if len(hit) else "no build-up starts at that time any more (check the clip)"
    print(f"{name:<16} {r['action']:<15} window={r.get('window')}  ->  {state}")


b164_ATM_43-22   stoppage        window=('43:26.9', '43:29.9')  ->  143 dead_or_untracked out_or_stoppage_then_opponent
b169_ATM_44-17   stoppage        window=('44:25.1', '44:28.1')  ->  147 dead_or_untracked same_team_after_gap
b114_ATM_30-20   fix_possession  window=('30:42.8', '30:44.4')  ->  101 success entered_attacking_third
b126_ATM_34-46   fix_possession  window=None  ->  112 lost turnover_to_opponent
b137_ATM_36-54   fix_possession  window=('37:02.0', '37:04.4')  ->  123 lost turnover_to_opponent
b155_BAR_40-37   fix_possession  window=None  ->  135 lost turnover_to_opponent
b027_BAR_07-43   fix_possession  window=('08:06.2', '08:09.8')  ->  27 success entered_attacking_third
b047_BAR_13-17   fix_possession  window=('13:28.0', '13:30.2')  ->  45 success entered_attacking_third
b149_BAR_39-27   fix_possession  window=('39:40.8', '39:44.3')  ->  no build-up starts at that time any more (check the clip)
b161_BAR_42-29   fix_possession  window=('42:34.4', '42:36.3')  ->  141 succ

### 5d. Compare with the 2 s snapshot
Compares this run (merge gap 5 s + manual fixes) with `buildups_snapshot_merge2s.parquet` saved from the 2 s run, using your verdicts on the dead clips. Skip it if the snapshot file does not exist.

In [18]:
# CELL C — run AFTER the re-run with the new gap (put it after 5c). Compares with the snapshot and your review verdicts.
# =====================================================================================================================
import re
SNAP_PATH = OUT_DIR / "buildups_snapshot_merge2s.parquet"
if not SNAP_PATH.exists():
    raise FileNotFoundError(f"no snapshot at {SNAP_PATH}: this cell compares with the 2 s run saved earlier. Skip it if you do not have it.")
# your verdicts on the dead clips: "dead" = you confirmed it is dead, "review" = you said it needs review / is not dead
VERDICTS = {k: "dead" for k in """
b008_ATM_02-42 b018_ATM_05-10 b019_ATM_05-19 b020_ATM_05-32 b025_ATM_07-02 b037_ATM_11-05 b039_BAR_11-39 b042_ATM_12-14
b055_ATM_16-27 b058_ATM_16-45 b060_BAR_17-07 b061_BAR_17-13 b062_BAR_17-59 b071_ATM_19-48 b072_ATM_19-58 b082_ATM_22-12
b083_ATM_22-43 b088_ATM_24-54 b090_BAR_25-39 b094_ATM_26-35 b101_BAR_28-04 b103_BAR_28-25 b103_BAR_28-33 b108_ATM_29-32
b111_BAR_30-49 b136_BAR_38-15 b140_BAR_39-20 b143_ATM_40-10 b153_ATM_43-05 b162_BAR_44-53 b168_BAR_46-25
""".split()}
VERDICTS.update({k: "review" for k in """
b002_ATM_01-06 b010_ATM_03-22 b043_ATM_12-31 b063_BAR_18-18 b076_BAR_21-20 b084_ATM_22-59 b099_ATM_27-43 b105_ATM_28-46
b113_BAR_31-59 b116_BAR_33-00 b121_ATM_34-46 b133_ATM_37-26 b137_BAR_38-24 b154_ATM_43-22 b158_ATM_44-17
""".split()})

def _key(clip):
    m = re.search(r"b?\d+_([A-Za-z]+)_(\d+)-(\d+)", clip)
    return (m.group(1), f"{int(m.group(2)):02d}:{int(m.group(3)):02d}")
VERDICT_BY_KEY = {_key(k): (k, v) for k, v in VERDICTS.items()}

old = pd.read_parquet(SNAP_PATH)
new = buildups[["buildup_id", "team_name", "start_time", "start_frame", "end_frame", "outcome", "end_detail"]].copy()

def match_new(r):
    m = new[(new.team_name == r.team_name) & (new.start_frame <= r.start_frame) & (new.end_frame >= r.start_frame)]
    return m.iloc[0] if len(m) else None

rows = []
for r in old.itertuples():
    n = match_new(r)
    clip, verdict = VERDICT_BY_KEY.get((r.team_name, r.start_time), (None, ""))
    rows.append(dict(old_id=r.buildup_id, team=r.team_name, start=r.start_time, old_outcome=r.outcome, old_detail=r.end_detail,
                     new_id=None if n is None else int(n.buildup_id), new_outcome="(gone)" if n is None else n.outcome,
                     new_detail="" if n is None else n.end_detail, verdict=verdict, clip=clip))
cmp_ = pd.DataFrame(rows)
cmp_["changed"] = (cmp_.old_outcome != cmp_.new_outcome) | (cmp_.old_detail != cmp_.new_detail)

d = cmp_[cmp_.old_outcome == "dead_or_untracked"]
print(f"build-ups: {len(old)} (2 s) -> {len(new)} (new gap) | dead_or_untracked: {len(d)} -> {int((new.outcome == 'dead_or_untracked').sum())}")
print("\nold dead -> new outcome:")
print(d.new_outcome.value_counts().to_string())

print("\n=== OLD DEAD THAT YOU CONFIRMED AS DEAD, but changed (these must be checked) ===")
x = d[(d.verdict == "dead") & d.changed]
print(x[["clip", "old_id", "new_id", "new_outcome", "new_detail"]].to_string(index=False) if len(x) else "none")

print("\n=== OLD DEAD THAT YOU SAID NEED REVIEW: what they are now ===")
x = d[d.verdict == "review"]
print(x[["clip", "old_id", "new_id", "new_outcome", "new_detail", "changed"]].to_string(index=False))

print("\n=== OLD DEAD WITHOUT A VERDICT, changed ===")
x = d[(d.verdict == "") & d.changed]
print(x[["team", "start", "old_id", "new_id", "new_outcome", "new_detail"]].to_string(index=False) if len(x) else "none")

print("\n=== side effects: build-ups that were NOT dead before and changed outcome ===")
x = cmp_[(cmp_.old_outcome != "dead_or_untracked") & (cmp_.old_outcome != cmp_.new_outcome)]
print(x[["team", "start", "old_id", "new_id", "old_outcome", "new_outcome", "new_detail"]].to_string(index=False) if len(x) else "none")

print("\nbuild-ups that are dead now but were not dead before (render them in the check cell):")
newdead = new[new.outcome == "dead_or_untracked"]
fresh = [r for r in newdead.itertuples() if not ((old.team_name == r.team_name) & (old.start_frame <= r.start_frame) & (old.end_frame >= r.start_frame) & (old.outcome == "dead_or_untracked")).any()]
print([f"b{r.buildup_id:03d}_{r.team_name}_{r.start_time.replace(':', '-')}" for r in fresh] or "none")
cmp_.to_csv(OUT_DIR / "merge_gap_comparison.csv", index=False)
print("\nfull table saved:", OUT_DIR / "merge_gap_comparison.csv")


build-ups: 166 (2 s) -> 156 (new gap) | dead_or_untracked: 46 -> 30

old dead -> new outcome:
new_outcome
dead_or_untracked    31
lost                  7
success               7
(gone)                1

=== OLD DEAD THAT YOU CONFIRMED AS DEAD, but changed (these must be checked) ===
          clip  old_id  new_id       new_outcome                    new_detail
b019_ATM_05-19      19    21.0              lost          turnover_to_opponent
b020_ATM_05-32      20    21.0              lost          turnover_to_opponent
b055_ATM_16-27      55    55.0           success       entered_attacking_third
b058_ATM_16-45      58    57.0              lost          turnover_to_opponent
b060_BAR_17-07      60    58.0 dead_or_untracked out_or_stoppage_then_opponent
b088_ATM_24-54      88    82.0           success       entered_attacking_third
b140_BAR_39-20     140   130.0           success       entered_attacking_third

=== OLD DEAD THAT YOU SAID NEED REVIEW: what they are now ===
          clip  old_i

In [19]:
print(P["MERGE_GAP_SEC"], len(spells), len(buildups))

2.0 167 156


## 6. Pressure measurement
Smooths tracks, computes each defender's distance and closing speed to the ball carrier, and flags pressing defenders and sustained "under pressure" frames.

In [ ]:
P.update(
    SMOOTH_WINDOW_FRAMES=9,        # Savitzky-Golay window (odd), ~0.36 s
    SMOOTH_POLY=2,
    MAX_FILL_FRAMES=5,             # interpolate tracking gaps up to this long
    MAX_SPEED_MPS=9.5,             # faster than this = tracking jump, ignored
    PRESS_DIST_M=5.0,              # a defender presses if within this ...
    PRESS_TIGHT_M=2.0,             # ... and either this close, or
    CLOSING_WINDOW_SEC=0.5,        #     closing in at >= CLOSING_SPEED_MPS (measured over this window)
    CLOSING_SPEED_MPS=1.5,
    NEAR_DIST_M=7.0,               # "near" defenders (context only)
    PRESS_SUSTAIN_SEC=0.4,         # pressing must last this long in a row to count as "under pressure"
)

def smooth_tracks(pl):
    win, poly = P["SMOOTH_WINDOW_FRAMES"], P["SMOOTH_POLY"]
    team_of = pl.groupby("track_id")["team"].agg(lambda s: s.mode().iloc[0])
    parts = []
    for tid, g in pl.groupby("track_id", sort=False):
        g = g.drop_duplicates("frame_idx").sort_values("frame_idx")
        full = np.arange(g.frame_idx.iloc[0], g.frame_idx.iloc[-1] + 1)
        s = g.set_index("frame_idx").reindex(full)
        x = s["x"].interpolate(limit=P["MAX_FILL_FRAMES"], limit_area="inside").to_numpy(float)
        y = s["y"].interpolate(limit=P["MAX_FILL_FRAMES"], limit_area="inside").to_numpy(float)
        valid = ~(np.isnan(x) | np.isnan(y))
        xs, ys = x.copy(), y.copy()
        for a, b in bool_runs(valid, full):
            if b - a + 1 >= win:
                xs[a:b + 1] = savgol_filter(x[a:b + 1], win, poly)
                ys[a:b + 1] = savgol_filter(y[a:b + 1], win, poly)
        parts.append(pd.DataFrame({"frame_idx": full[valid], "track_id": tid, "x": xs[valid], "y": ys[valid]}))
    out = pd.concat(parts, ignore_index=True)
    out["team"] = out["track_id"].map(team_of).astype(int)
    return out

pf = smooth_tracks(player[player.role.isin(OUTFIELD_ROLES)])

# press target = the possession location from cell 6 (carrier, else ball), in raw pitch coordinates
tg = loc[(loc.spell_id >= 0) & loc.x.notna()][["frame_idx", "spell_id", "poss_team", "x", "y", "src"]].rename(columns={"x": "tx", "y": "ty"})
tg["def_team"] = 1 - tg["poss_team"].astype(int)

d = pf.merge(tg[["frame_idx", "spell_id", "def_team", "tx", "ty"]], on="frame_idx")
d = d[d["team"] == d["def_team"]].copy()                      # only the defending team's outfield players
d["dist"] = np.hypot(d["x"] - d["tx"], d["y"] - d["ty"])

# closing speed = how fast the defender's OWN movement brings him towards the target's current position
W = F(P["CLOSING_WINDOW_SEC"])
d = d.sort_values(["track_id", "frame_idx"]).reset_index(drop=True)
g_ = d.groupby("track_id")
px, py, pfr = g_["x"].shift(W), g_["y"].shift(W), g_["frame_idx"].shift(W)
contiguous = (d["frame_idx"] - pfr) == W
d["closing"] = np.where(contiguous, (np.hypot(px - d["tx"], py - d["ty"]) - d["dist"]) / (W / FPS), np.nan)
glitch = d["closing"].abs() > P["MAX_SPEED_MPS"]
n_glitch = int(glitch.sum())
d.loc[glitch, "closing"] = np.nan

d["is_press"] = (d["dist"] < P["PRESS_DIST_M"]) & ((d["dist"] < P["PRESS_TIGHT_M"]) | (d["closing"] >= P["CLOSING_SPEED_MPS"]))
d["is_near"] = d["dist"] < P["NEAR_DIST_M"]

pa = (d.groupby("frame_idx")
      .agg(n_pressing=("is_press", "sum"), n_near=("is_near", "sum"), min_dist=("dist", "min"), n_def_tracked=("dist", "size")))
pr = tg[["frame_idx", "spell_id", "poss_team", "def_team", "src"]].merge(pa, on="frame_idx", how="left")
pr[["n_pressing", "n_near", "n_def_tracked"]] = pr[["n_pressing", "n_near", "n_def_tracked"]].fillna(0).astype(int)

# "under pressure" = at least one pressing defender, held for PRESS_SUSTAIN_SEC in a row (inside one spell)
pr = pr.sort_values("frame_idx").reset_index(drop=True)
pr["under_pressure"] = False
sustain = F(P["PRESS_SUSTAIN_SEC"])
for sid, idx in pr.groupby("spell_id").indices.items():
    frames = pr["frame_idx"].to_numpy()[idx]
    for s, e in bool_runs(pr["n_pressing"].to_numpy()[idx] >= 1, frames):
        if e - s + 1 >= sustain:
            pr.loc[idx[s:e + 1], "under_pressure"] = True

# ------------------------------------------------------------------ checks
print(f"{len(pf):,} smoothed outfield player-frames; {len(d):,} defender-frames in possession frames with a target")
print(f"closing speed available for {d.closing.notna().mean() * 100:.1f}%; {n_glitch:,} values above {P['MAX_SPEED_MPS']} m/s ignored as tracking jumps")
print("distance to target, quantiles (m):", d["dist"].quantile([0.05, 0.25, 0.5, 0.75]).round(1).to_dict())

pr["pressed_team"] = pr["poss_team"].astype(int).map(TEAM_NAMES)    # the team that HAS the ball and is being pressed
print(f"\nframes with a target: {len(pr):,}")
print("\nshare of frames (%) with >=1 pressing defender (raw) and 'under pressure' (sustained), by pressed team:")
print(pr.groupby("pressed_team")[["n_pressing", "under_pressure"]].agg(lambda s: (s >= 1).mean() * 100 if s.name == "n_pressing" else s.mean() * 100)
      .rename(columns={"n_pressing": "raw_%", "under_pressure": "sustained_%"}).round(1).to_string())

print("\nunder pressure (%) by where the target comes from:")
print((pr.groupby("src")["under_pressure"].mean() * 100).round(1).to_string())
print("\nnumber of pressing defenders when under pressure:")
print(pr.loc[pr.under_pressure, "n_pressing"].value_counts().sort_index().head(6).to_string())
print("\ndefenders tracked per frame (outfield, defending team):", pr["n_def_tracked"].describe().round(1)[["mean", "min", "25%", "50%", "75%", "max"]].to_dict())
print(f"frames with fewer than 6 defenders tracked: {(pr['n_def_tracked'] < 6).mean() * 100:.1f}%")

## 7. Pressure features per build-up
Time to first pressure, number of pressers, pressure share, where and on which side the first pressure happens. Produces `buildups_p`.

In [ ]:
P["PRESSED_AT_START_SEC"] = 1.0    # pressure within this long of the start = "pressed from the start" (typically a counter-press)
P["PRESS_AT_END_SEC"] = 1.0        # pressure in the last stretch before the build-up ends

pr_i = pr.set_index("frame_idx")            # frames that have a press target (one row per frame)
loc_i = loc.set_index("frame_idx")
start_f, end_f = F(P["PRESSED_AT_START_SEC"]), F(P["PRESS_AT_END_SEC"])

def channel_of(y_att):
    """Neutral names until we confirm on video which side of the pitch is 'left' (y_low / y_high in the team's attacking frame)."""
    return "y_low" if y_att < PITCH_WIDTH / 3 else ("y_high" if y_att > 2 * PITCH_WIDTH / 3 else "centre")

rows = []
for b in buildups.itertuples():
    seg = pr_i.loc[b.start_frame:b.end_frame]
    n_total = b.end_frame - b.start_frame + 1
    r = dict(buildup_id=b.buildup_id, target_coverage=len(seg) / n_total)
    if len(seg) == 0:
        rows.append(r); continue
    up = seg["under_pressure"].to_numpy(bool); fr_ = seg.index.to_numpy()
    pressed = bool(up.any())
    r.update(
        pressed_any=pressed,
        pressure_share=float(up.mean()),
        n_pressure_episodes=len(bool_runs(up, fr_)),
        max_pressers=int(seg["n_pressing"].max()),
        min_dist_m=float(seg["min_dist"].min()),
        ball_target_share=float((seg["src"] != "carrier").mean()),
        pressed_at_end=bool(up[fr_ > b.end_frame - end_f].any()),
    )
    if pressed:
        first = int(fr_[np.argmax(up)])
        xa, ya = loc_i.loc[first, "x_att"], loc_i.loc[first, "y_att"]
        r.update(
            time_to_first_pressure_sec=(first - b.start_frame) / FPS,
            pressed_from_start=bool(first - b.start_frame <= start_f),
            first_press_frame=first,
            first_press_third=third_of(xa) if pd.notna(xa) else None,
            first_press_channel=channel_of(ya) if pd.notna(ya) else None,
            mean_pressers_when_pressed=float(seg.loc[up, "n_pressing"].mean()),
        )
    else:
        r.update(pressed_from_start=False)
    rows.append(r)

buildups_p = buildups.merge(pd.DataFrame(rows), on="buildup_id", how="left")

# ------------------------------------------------------------------ checks
print(f"{len(buildups_p)} build-ups with pressure features")
print(f"target coverage (share of a build-up's frames with a press target): "
      f"median {buildups_p.target_coverage.median() * 100:.0f}%, build-ups below 50%: {(buildups_p.target_coverage < 0.5).sum()}")

def pct(s): return round(float(s.mean() * 100), 1)

print("\nby team:")
t = buildups_p.groupby("team_name").agg(
    buildups=("buildup_id", "size"),
    pressed_any_pct=("pressed_any", pct),
    pressed_from_start_pct=("pressed_from_start", pct),
    median_time_to_first_pressure_s=("time_to_first_pressure_sec", "median"),
    mean_pressure_share_pct=("pressure_share", lambda s: round(float(s.mean() * 100), 1)),
    mean_max_pressers=("max_pressers", "mean"))
print(t.round(2).to_string())

print("\nby outcome (descriptive only: longer build-ups have more time to be pressed):")
o = buildups_p.groupby(["team_name", "outcome"]).agg(
    n=("buildup_id", "size"), pressed_any_pct=("pressed_any", pct), pressed_at_end_pct=("pressed_at_end", pct),
    pressed_from_start_pct=("pressed_from_start", pct), median_duration_s=("duration_sec", "median"))
print(o.round(1).to_string())

print("\nwhere the first pressure happens (third, in the pressed team's attacking frame), count:")
print(buildups_p.groupby(["team_name", "first_press_third"]).size().unstack(fill_value=0).to_string())
print("\nfirst pressure channel (neutral names), count:")
print(buildups_p.groupby(["team_name", "first_press_channel"]).size().unstack(fill_value=0).to_string())

print("\ntime to first pressure (s), quantiles:", buildups_p["time_to_first_pressure_sec"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_dict())
print(f"build-ups where most frames are on the ball (target not the carrier): {(buildups_p.ball_target_share > 0.5).sum()}")
print()
print(buildups_p.head(8)[["buildup_id", "team_name", "duration_sec", "outcome", "pressed_any", "time_to_first_pressure_sec",
                          "pressed_from_start", "pressure_share", "max_pressers", "pressed_at_end", "first_press_third"]].round(2).to_string())

## 8. Pressed vs not pressed: outcome
Among build-ups still alive 3 s after the start, does the later outcome differ if they were pressed? Wilson intervals and Fisher exact test.

In [ ]:
from scipy.stats import fisher_exact

P["LANDMARK_SEC"] = 3.0     # compare build-ups that are still alive this long after their start
P["GRACE_SEC"] = 1.0        # ignore pressure in the first second: at a turnover the player who just lost the ball is next to the new carrier

def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n; den = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - h, c + h

def landmark_table(bp, landmark_sec, grace_sec):
    """Build-ups still alive at the landmark, split by whether they were pressed between the grace period and the landmark.
    Everyone in the sample is still 'at risk' of success / loss afterwards, so pressed vs not is a fair comparison."""
    L, G = F(landmark_sec), F(grace_sec)
    rows = []
    for b in bp.itertuples():
        if b.end_frame <= b.start_frame + L:           # ended before the landmark: not comparable
            continue
        seg = pr_i.loc[b.start_frame + G: b.start_frame + L]
        if len(seg) == 0:
            continue
        rows.append(dict(buildup_id=b.buildup_id, team_name=b.team_name, outcome=b.outcome,
                         pressed=bool(seg["under_pressure"].any())))
    return pd.DataFrame(rows)

def summarize(lt):
    out = []
    for team, g in list(lt.groupby("team_name")) + [("BOTH", lt)]:
        for pressed in (True, False):
            gg = g[g.pressed == pressed]
            n = len(gg); s = int((gg.outcome == "success").sum()); l = int((gg.outcome == "lost").sum())
            dec = s + l; lo, hi = wilson(s, dec)
            out.append(dict(team=team, pressed=pressed, alive_at_landmark=n, success=s, lost=l, undecided=n - dec,
                            success_of_decided_pct=round(100 * s / dec, 1) if dec else np.nan,
                            ci95=f"{lo * 100:.0f}-{hi * 100:.0f}" if dec else "n/a"))
    return pd.DataFrame(out)

lt = landmark_table(buildups_p, P["LANDMARK_SEC"], P["GRACE_SEC"])
tab = summarize(lt)
print(f"landmark = {P['LANDMARK_SEC']} s after the start, pressure counted from {P['GRACE_SEC']} s to the landmark")
print(f"{len(lt)} of {len(buildups_p)} build-ups are still alive at the landmark\n")
print("pressed vs not pressed, outcome AFTER the landmark (success_of_decided = success / (success + lost)):")
print(tab.to_string(index=False))

print("\nFisher exact test, pressed vs not pressed (success vs lost among decided), p-value:")
for team, g in list(lt.groupby("team_name")) + [("BOTH", lt)]:
    t = [[int(((g.pressed == p) & (g.outcome == o)).sum()) for o in ("success", "lost")] for p in (True, False)]
    p_val = fisher_exact(t)[1] if min(map(sum, t)) > 0 else np.nan
    print(f"  {team}: pressed {t[0]} / not pressed {t[1]}  (success, lost)  p = {p_val:.2f}")

rows = []
for L_ in [2.0, 3.0, 4.0, 5.0]:
    for G_ in [0.0, 1.0]:
        s_ = summarize(landmark_table(buildups_p, L_, G_))
        s_ = s_[s_.team == "BOTH"].set_index("pressed")
        rows.append(dict(landmark_s=L_, grace_s=G_, n_pressed=s_.loc[True, "alive_at_landmark"], n_not=s_.loc[False, "alive_at_landmark"],
                         success_pressed_pct=s_.loc[True, "success_of_decided_pct"], success_not_pressed_pct=s_.loc[False, "success_of_decided_pct"]))
print("\nsensitivity to the landmark and the grace period (both teams combined):")
print(pd.DataFrame(rows).to_string(index=False))

## 9. Loss hazard within 2 s
One moment per second inside each build-up: chance of losing the ball within 2 s when pressed in the last second vs not. Bootstrap over build-ups for the interval.

In [ ]:
P["HAZ_GRACE_SEC"] = 1.0      # skip the first second (the turnover contest)
P["HAZ_LOOKBACK_SEC"] = 1.0   # "pressed" = under pressure at some point in the last second
P["HAZ_HORIZON_SEC"] = 2.0    # outcome = ball lost within this long after the moment
P["HAZ_STEP_SEC"] = 1.0       # one moment per second
N_BOOT = 2000

G_, Lb, H_, S_ = F(P["HAZ_GRACE_SEC"]), F(P["HAZ_LOOKBACK_SEC"]), F(P["HAZ_HORIZON_SEC"]), F(P["HAZ_STEP_SEC"])
rows = []
for b in buildups_p.itertuples():
    t = b.start_frame + G_ + Lb
    while t < b.end_frame:
        seg = pr_i.loc[t - Lb + 1: t]
        if len(seg):
            censored = (b.outcome in ("dead_or_untracked", "data_end")) and (b.end_frame <= t + H_)
            if not censored:
                up = seg["under_pressure"]
                rows.append(dict(buildup_id=b.buildup_id, team_name=b.team_name, pressed=bool(up.any()),
                                 max_pressers=int(seg.loc[up, "n_pressing"].max()) if up.any() else 0,
                                 carrier_share=float((seg["src"] == "carrier").mean()),
                                 loss_next=bool(b.outcome == "lost" and b.end_frame <= t + H_)))
        t += S_
win = pd.DataFrame(rows)

def boot_table(df, label, seed=1):
    c = (df.assign(pn=df.pressed, pl=df.pressed & df.loss_next, un=~df.pressed, ul=(~df.pressed) & df.loss_next)
           .groupby("buildup_id")[["pn", "pl", "un", "ul"]].sum())
    a = c.to_numpy(float); tot = a.sum(axis=0)
    rate = lambda l, n: 100 * l / n if n else np.nan
    rng = np.random.default_rng(seed)
    s = a[rng.integers(0, len(a), (N_BOOT, len(a)))].sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        diff = 100 * (s[:, 1] / s[:, 0] - s[:, 3] / s[:, 2])
    lo, hi = np.nanpercentile(diff, [2.5, 97.5])
    return dict(group=label, buildups=len(c),
                pressed_windows=int(tot[0]), pressed_losses=int(tot[1]), pressed_loss_pct=round(rate(tot[1], tot[0]), 1),
                unpressed_windows=int(tot[2]), unpressed_losses=int(tot[3]), unpressed_loss_pct=round(rate(tot[3], tot[2]), 1),
                diff_pp=round(rate(tot[1], tot[0]) - rate(tot[3], tot[2]), 1), ci95=f"{lo:.1f} to {hi:.1f}")

print(f"{len(win):,} one-second moments from {win.buildup_id.nunique()} build-ups; "
      f"{int(win.loss_next.sum())} lost the ball within {P['HAZ_HORIZON_SEC']:.0f} s; "
      f"{win.pressed.mean() * 100:.0f}% of moments were pressed")

print(f"\nchance of losing the ball within {P['HAZ_HORIZON_SEC']:.0f} s, pressed in the last {P['HAZ_LOOKBACK_SEC']:.0f} s vs not (diff in percentage points, 95% interval from resampling build-ups):")
tab = pd.DataFrame([boot_table(win[win.team_name == t], t) for t in sorted(win.team_name.unique())] + [boot_table(win, "BOTH")])
print(tab.to_string(index=False))

print("\nsame, only moments where most of the last second was measured on a carrier (not the ball):")
wc = win[win.carrier_share >= 0.5]
print(pd.DataFrame([boot_table(wc[wc.team_name == t], t) for t in sorted(wc.team_name.unique())] + [boot_table(wc, "BOTH")]).to_string(index=False))

win["pressers"] = np.select([win.max_pressers == 0, win.max_pressers == 1], ["none", "1 presser"], "2+ pressers")
lvl = (win.groupby(["team_name", "pressers"]).agg(moments=("loss_next", "size"), losses=("loss_next", "sum")))
lvl["loss_pct"] = (100 * lvl.losses / lvl.moments).round(1)
print("\nloss rate by number of pressers in the last second:")
print(lvl.to_string())

## 10. Video setup
OpenCV, video path and frame offset, base annotation colours. Only needed for sections 12–14.

In [ ]:
import importlib.util
HAVE_CV2 = importlib.util.find_spec("cv2") is not None
if HAVE_CV2:
    import cv2
    print("OpenCV", cv2.__version__)
else:
    print("OpenCV is missing: run  pip install opencv-python  in this environment, then restart the kernel")

VIDEO_PATH = "C:/Users/user/Desktop/Football_cv_project/football-pipeline (1)/project/barca_atletico_first_half/clip.mkv"
FRAME_OFFSET = 0           # video frame = frame_idx + FRAME_OFFSET   (change it if the drawings are shifted in time)

# ------------------------------------------------------------------ base annotation style (used for ALL clips)
STYLE = dict(
    team_bgr={0: (40, 40, 230), 1: (200, 90, 0)},   # team 0 (ATM) red, team 1 (BAR) blue   (BGR)
    referee_bgr=(70, 70, 70),
    ball_bgr=(0, 255, 255),                         # yellow ball circle
    carrier_bgr=(255, 255, 255),                    # inverted triangle above the ball carrier
)
ALLOWED_BALL_SOURCES = None                         # None = draw the ball in EVERY frame that has a ball position (detected AND interpolated), always as the same yellow circle
ALLOWED_PLAYER_SOURCES = None                       # None = draw every tracked player; set e.g. {"tracked"} after we see the counts

def draw_base_layer(img, players, ball_xy, carrier_id):
    """Player ellipse at the feet + number tag in the team colour, inverted triangle above the carrier, yellow ball circle."""
    for r in players.itertuples():
        if pd.isna(r.x1) or pd.isna(r.foot_px_x):
            continue
        is_ref = (r.role == "referee")
        col = STYLE["referee_bgr"] if is_ref else STYLE["team_bgr"].get(int(r.team), (128, 128, 128))
        fx, fy, w = int(r.foot_px_x), int(r.foot_px_y), max(int(r.x2 - r.x1), 16)
        cv2.ellipse(img, (fx, fy), (int(w * 0.7), int(w * 0.25)), 0, 0, 360, col, 3, cv2.LINE_AA)    # position marker at the feet
        if is_ref:
            continue
        label = str(int(r.number)) if pd.notna(r.number) else "?"
        cx, top = int((r.x1 + r.x2) / 2), int(r.y1)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(img, (cx - tw // 2 - 5, top - th - 12), (cx + tw // 2 + 5, top - 4), col, -1)    # number tag
        cv2.putText(img, label, (cx - tw // 2, top - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        if pd.notna(carrier_id) and r.track_id == carrier_id:                                       # flipped triangle = ball carrier
            ty = top - th - 16
            tri = np.array([[cx - 11, ty - 17], [cx + 11, ty - 17], [cx, ty]], np.int32)
            cv2.fillPoly(img, [tri], STYLE["carrier_bgr"]); cv2.polylines(img, [tri], True, (0, 0, 0), 2, cv2.LINE_AA)
    if ball_xy is not None:
        cv2.circle(img, (int(ball_xy[0]), int(ball_xy[1])), 12, STYLE["ball_bgr"], 3, cv2.LINE_AA)
    return img

if HAVE_CV2:
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
    v_fps = cap.get(cv2.CAP_PROP_FPS); v_n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    v_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); v_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"\nvideo: {v_w}x{v_h}, {v_fps:.2f} fps, {v_n:,} frames ({v_n / max(v_fps, 1) / 60:.1f} min)")
    print(f"tracking: {int(ball.frame_idx.max()) + 1:,} frames at {FPS} fps -> "
          f"{'frame counts match' if abs(v_n - (int(ball.frame_idx.max()) + 1)) <= 2 else 'frame counts DIFFER, check the offset / fps'}")

    # what the 'source' columns contain (for information: every ball position is drawn the same way)
    ps = pd.read_parquet(PLAYER_PATH, columns=["source", "conf"])
    bs = pd.read_parquet(BALL_PATH, columns=["ball_source", "ball_conf", "ball_px_x"])
    print("\nplayer rows by source:", ps["source"].value_counts(dropna=False).to_dict())
    print("ball frames by ball_source (with a pixel position):", bs[bs.ball_px_x.notna()]["ball_source"].value_counts(dropna=False).to_dict())
    print("ball frames by ball_source (all):", bs["ball_source"].value_counts(dropna=False).to_dict())
    del ps, bs

    # a lost build-up that was pressed at the end, mostly measured on a carrier: three moments to check the drawing against the picture
    cand = buildups_p[(buildups_p.outcome == "lost") & (buildups_p.pressed_at_end == True) &
                      (buildups_p.target_coverage > 0.8) & (buildups_p.ball_target_share < 0.3) & (buildups_p.duration_sec > 4)]
    b = cand.iloc[len(cand) // 2]
    first = int(b.first_press_frame) if pd.notna(b.first_press_frame) else int(b.start_frame)
    test_frames = [first, int((first + b.end_frame) // 2), int(b.end_frame)]
    print(f"\nchecking build-up {int(b.buildup_id)} ({b.team_name}, lost, starts {b.start_time}) at frames {test_frames}")

    pcols = ["frame_idx", "track_id", "role", "team", "number", "x1", "y1", "x2", "y2", "foot_px_x", "foot_px_y", "source"]
    try:
        boxes = pd.read_parquet(PLAYER_PATH, columns=pcols, filters=[("frame_idx", "in", test_frames)])
    except Exception:
        boxes = pd.read_parquet(PLAYER_PATH, columns=pcols); boxes = boxes[boxes.frame_idx.isin(test_frames)]
    if ALLOWED_PLAYER_SOURCES is not None:
        boxes = boxes[boxes.source.isin(ALLOWED_PLAYER_SOURCES)]
    bpx = pd.read_parquet(BALL_PATH, columns=["frame_idx", "ball_px_x", "ball_px_y", "ball_source", "carrier_track_id"])
    bpx = bpx[bpx.frame_idx.isin(test_frames)].set_index("frame_idx")

    fig, axes = plt.subplots(1, 3, figsize=(24, 5.2))
    for ax, f in zip(axes, test_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, f + FRAME_OFFSET)
        ok_, img = cap.read()
        if not ok_:
            ax.set_title(f"frame {f}: could not read"); ax.axis("off"); continue
        row = bpx.loc[f]
        ball_xy = (row.ball_px_x, row.ball_px_y) if (pd.notna(row.ball_px_x) and (ALLOWED_BALL_SOURCES is None or row.ball_source in ALLOWED_BALL_SOURCES)) else None
        draw_base_layer(img, boxes[boxes.frame_idx == f], ball_xy, row.carrier_track_id)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"frame {f}  ({mmss(f)})  ball: {('drawn, source = ' + str(row.ball_source)) if ball_xy else 'no ball position in this frame'}"); ax.axis("off")
    plt.tight_layout()
    out_png = OUT_DIR / "alignment_check.png"
    plt.savefig(out_png, dpi=80); plt.show()
    print("saved", out_png)
    cap.release()

## 11. Roster & inferred positions
Number, name and position per player. Positions are **inferred** from average location; correct wrong ones with `POSITION_OVERRIDE`. Needed by the clips and by the LLM prompts.

In [ ]:
# ================================================================== roster: number, name and POSITION for every player
# The tables have no tactical position, so it is INFERRED from each player's average location (own attacking frame):
# outfield players are split into defenders / midfielders / forwards, then labelled left to right inside each line
# using only: defenders LB CB RB, midfielders LM CM RM, forwards LW ST RW (wide labels only for players standing wide).
LEFT_IS_LOW_Y_ATT = True        # in a team's attacking frame, is low y its LEFT side? (flip if left/right look mirrored on the video)
POSITION_OVERRIDE = {204: "LM", 222: "RM", 117: "CM"}   # manual fixes by track_id: 204 Gallagher, 222 Simeone, 117 Casado (delete a pair to undo it)
WIDE_FROM_TOUCHLINE_M = 18.0    # a player counts as "wide" (LB/RB, LM/RM, LW/RW) only if he stands within this distance of a touchline

ros = pd.read_parquet(PLAYER_PATH, columns=["track_id", "team", "role", "name", "number"]).dropna(subset=["team"])
ros["team"] = ros["team"].astype(int); ros["track_id"] = ros["track_id"].astype(int)
mode_ = lambda s: s.dropna().mode().iloc[0] if s.notna().any() else np.nan
roster = (ros.groupby("track_id").agg(team=("team", "first"), role=("role", mode_), name=("name", mode_), number=("number", mode_))
          .reset_index())
med = player.groupby("track_id")[["x", "y"]].median()
roster = roster.merge(med, left_on="track_id", right_index=True, how="left")
att_pos = roster["team"].map(TEAM_ATTACKS_POSITIVE_X).astype(bool)
roster["x_att"] = np.where(att_pos, roster["x"], PITCH_LENGTH - roster["x"])
roster["y_att"] = np.where(att_pos, roster["y"], PITCH_WIDTH - roster["y"])

# allowed labels per line: (left-wide, central, right-wide)
WIDE = {"D": ("LB", "CB", "RB"), "M": ("LM", "CM", "RM"), "F": ("LW", "ST", "RW")}

def label_line(part, line):
    """Left to right inside one line. The outer player on a side gets the wide label (LB/LM/LW or RB/RM/RW) only if he really stands
    within WIDE_FROM_TOUCHLINE_M of a touchline; everyone else gets the central label (CB / CM / ST). A back three is three CBs."""
    left_lab, mid_lab, right_lab = WIDE[line]
    part = part.sort_values("y_att", ascending=LEFT_IS_LOW_Y_ATT)         # first = left side
    ys, n = part["y_att"].to_numpy(), len(part)
    labs = [mid_lab] * n
    if n >= 2 and not (line == "D" and n == 3):
        lo, hi = WIDE_FROM_TOUCHLINE_M, PITCH_WIDTH - WIDE_FROM_TOUCHLINE_M
        left_wide = ys[0] < lo if LEFT_IS_LOW_Y_ATT else ys[0] > hi
        right_wide = ys[-1] > hi if LEFT_IS_LOW_Y_ATT else ys[-1] < lo
        if left_wide:
            labs[0] = left_lab
        if right_wide:
            labs[-1] = right_lab
    return dict(zip(part.track_id.astype(int), labs))

def assign_positions(team_rows):
    out = team_rows[team_rows.role == "player"].sort_values("x_att")
    xs, n = out["x_att"].to_numpy(), len(out)
    best = None                                         # split into 3 lines (back / middle / front), sizes limited to football-like values
    for i in range(2, 6):
        for j in range(i + 1, n):
            nm, nf = j - i, n - j
            if not (2 <= nm <= 5 and 1 <= nf <= 4):
                continue
            sse = sum(((g - g.mean()) ** 2).sum() for g in (xs[:i], xs[i:j], xs[j:]))
            if best is None or sse < best[0]:
                best = (sse, i, j)
    if best is None:
        return {int(t): "?" for t in out.track_id}
    _, i, j = best
    pos = {}
    for line, part in zip("DMF", (out.iloc[:i], out.iloc[i:j], out.iloc[j:])):
        pos.update(label_line(part, line))
    return pos

POS_OF = {}
for team, g in roster.groupby("team"):
    POS_OF.update(assign_positions(g))
for r in roster.itertuples():
    if r.role == "goalkeeper":
        POS_OF[int(r.track_id)] = "GK"
POS_OF.update(POSITION_OVERRIDE)
roster["position"] = roster["track_id"].map(POS_OF)
NUM_OF = {int(r.track_id): (int(r.number) if pd.notna(r.number) else None) for r in roster.itertuples()}

print("roster with inferred positions (check them against the real line-up; fix with POSITION_OVERRIDE):")
for team, g in roster.groupby("team"):
    print(f"\n{TEAM_NAMES[team]} (team {team}), attacks towards {'+x' if TEAM_ATTACKS_POSITIVE_X[team] else '-x'}")
    g = g.assign(line=g.position.map({"GK": 0}).fillna(g.x_att)).sort_values(["line", "y_att"])
    print(g[["track_id", "number", "name", "position", "x_att", "y_att"]].round(1).to_string(index=False))

## 12. Annotation style (v5)
Broadcast-style tags (number + position), carrier marker and ball circle.

In [ ]:
# ================================================================== clean broadcast-style annotation, v5 (replaces draw_base_layer)
# - one fixed tag per player, anchored above the head: NUMBER + POSITION in the team colour. Tags never move or float:
#   positions are smoothed over time (smooth_boxes), so they do not jitter.
# - occlusion: tags are drawn far -> near, so a nearer player's tag sits on top of a farther one (like real depth).
#   If a far tag would be mostly covered, it switches to a compact number-only tag (readable, never moved).
# - carrier: white outline + small white flipped triangle above the tag.
# - ball: ONLY a small yellow circle, the same style whether the position is detected, predicted or interpolated.
from PIL import Image, ImageDraw, ImageFont
import matplotlib

FOOT_MARKER = True          # thin small footprint ellipse under each player (set False to remove)
LABEL_ALPHA = 232           # tag opacity 0-255
BALL_RADIUS_PX = 6          # small yellow ball circle (pixels at 1080p)
COMPACT_IF_COVERED = 0.25   # a far tag covered by more than this share of its area (by a nearer tag) becomes number-only
TEAM_RGB = {t: (b[2], b[1], b[0]) for t, b in STYLE["team_bgr"].items()}     # STYLE is BGR, PIL needs RGB

def _find_font(bold):
    cands = (["C:/Windows/Fonts/segoeuib.ttf", "C:/Windows/Fonts/arialbd.ttf"] if bold
             else ["C:/Windows/Fonts/segoeui.ttf", "C:/Windows/Fonts/arial.ttf"])
    cands.append(str(Path(matplotlib.get_data_path()) / "fonts" / "ttf" / ("DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf")))
    return next((c for c in cands if Path(c).exists()), None)

_FONT_PATH = {True: _find_font(True), False: _find_font(False)}
_font_cache = {}
def get_font(size, bold=False):
    key = (size, bold)
    if key not in _font_cache:
        p = _FONT_PATH[bold]
        _font_cache[key] = ImageFont.truetype(p, size) if p else ImageFont.load_default()
    return _font_cache[key]

def smooth_boxes(boxes, win=5):
    """Offline centred moving average of each player's box and feet position inside a clip, so tags stay put instead of jittering."""
    cols = ["x1", "y1", "x2", "y2", "foot_px_x", "foot_px_y"]
    b = boxes.sort_values(["track_id", "frame_idx"]).copy()
    seg = (b.groupby("track_id")["frame_idx"].diff() != 1).cumsum()           # never smooth across a gap in the frames
    for c in cols:
        b[c] = b.groupby(["track_id", seg])[c].transform(lambda s: s.rolling(win, center=True, min_periods=1).mean())
    return b

def _covered(rect, others):
    ax0, ay0, ax1, ay1 = rect; area = max((ax1 - ax0) * (ay1 - ay0), 1); best = 0.0
    for bx0, by0, bx1, by1 in others:
        w, h = min(ax1, bx1) - max(ax0, bx0), min(ay1, by1) - max(ay0, by0)
        if w > 0 and h > 0:
            best = max(best, w * h / area)
    return best

def draw_base_layer(img, players, ball_xy, carrier_id):
    H_, W_ = img.shape[:2]; s = H_ / 1080
    nf, pf_ = get_font(max(12, int(15 * s)), True), get_font(max(9, int(11 * s)), False)
    base = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).convert("RGBA")
    ov = Image.new("RGBA", base.size, (0, 0, 0, 0)); dr = ImageDraw.Draw(ov)
    pad_x, pad_y, gap = int(5 * s), int(2 * s), int(6 * s)
    items = []
    for r in players.itertuples():
        if pd.isna(r.x1) or pd.isna(r.foot_px_x):
            continue
        is_ref = (r.role == "referee")
        it = dict(r=r, ref=is_ref, col=(110, 110, 110) if is_ref else TEAM_RGB.get(int(r.team), (128, 128, 128)),
                  fx=int(r.foot_px_x), fy=int(r.foot_px_y), w=max(int(r.x2 - r.x1), 16))
        if not is_ref:
            num = str(int(r.number)) if pd.notna(r.number) else "?"
            pos = POS_OF.get(int(r.track_id), "")
            nw = dr.textlength(num, font=nf); pw = dr.textlength(pos, font=pf_) if pos else 0
            lh = int(nf.size + 2 * pad_y)
            it.update(num=num, pos=pos, nw=nw, lh=lh, lw_full=int(nw + (pw + 5 * s if pos else 0) + 2 * pad_x), lw_c=int(nw + 2 * pad_x),
                      cx=int((r.x1 + r.x2) / 2), top=int(r.y1))
        items.append(it)

    tags = [it for it in items if not it["ref"]]
    placed = []
    for it in sorted(tags, key=lambda i: -i["fy"]):             # nearest first: decide which far tags are covered
        lw, lh = it["lw_full"], it["lh"]
        rect = (it["cx"] - lw // 2, it["top"] - lh - gap, it["cx"] + lw // 2, it["top"] - gap)
        it["compact"] = bool(it["pos"]) and _covered(rect, placed) > COMPACT_IF_COVERED
        lw = it["lw_c"] if it["compact"] else it["lw_full"]
        it["rect"] = (int(np.clip(it["cx"] - lw // 2, 2, W_ - lw - 2)), max(it["top"] - lh - gap, 2), 0, 0)
        it["lw"] = lw
        placed.append((it["rect"][0], it["rect"][1], it["rect"][0] + lw, it["rect"][1] + lh))

    for it in sorted(items, key=lambda i: i["fy"]):             # draw far -> near: nearer tags end up on top
        col = it["col"]
        if FOOT_MARKER:
            dr.ellipse([it["fx"] - int(it["w"] * 0.45), it["fy"] - int(it["w"] * 0.14), it["fx"] + int(it["w"] * 0.45), it["fy"] + int(it["w"] * 0.14)],
                       outline=col + (225,), width=max(1, int(2 * s)))
        if it["ref"]:
            continue
        x0, y0 = it["rect"][0], it["rect"][1]; lw, lh = it["lw"], it["lh"]
        r = it["r"]; is_car = pd.notna(carrier_id) and r.track_id == carrier_id
        px_, py_ = x0 + lw // 2, y0 + lh
        dr.polygon([(px_ - int(4 * s), py_), (px_ + int(4 * s), py_), (px_, py_ + int(4 * s))], fill=col + (LABEL_ALPHA,))   # small pointer
        dr.rounded_rectangle([x0, y0, x0 + lw, y0 + lh], radius=int(lh * 0.28), fill=col + (LABEL_ALPHA,),
                             outline=(255, 255, 255, 240) if is_car else (255, 255, 255, 120), width=2 if is_car else 1)
        yc = y0 + lh // 2
        dr.text((x0 + pad_x, yc), it["num"], font=nf, fill=(255, 255, 255, 255), anchor="lm")
        if it["pos"] and not it["compact"]:
            dr.text((x0 + pad_x + it["nw"] + 5 * s, yc), it["pos"], font=pf_, fill=(238, 238, 238, 255), anchor="lm")
        if is_car:                                              # flipped triangle above the carrier's tag
            mx, ty = x0 + lw // 2, y0 - int(3 * s)
            dr.polygon([(mx - int(7 * s), ty - int(10 * s)), (mx + int(7 * s), ty - int(10 * s)), (mx, ty)], fill=(255, 255, 255, 255), outline=(0, 0, 0, 255))
    if ball_xy is not None:                                     # ball: ONE style for every position (detected, predicted or interpolated)
        bx, by, br = int(ball_xy[0]), int(ball_xy[1]), max(4, int(BALL_RADIUS_PX * s))
        dr.ellipse([bx - br, by - br, bx + br, by + br], outline=(255, 230, 0, 255), width=max(2, int(2 * s)))
    img[:] = cv2.cvtColor(np.array(Image.alpha_composite(base, ov).convert("RGB")), cv2.COLOR_RGB2BGR)
    return img

print("ready:", "Image" in globals(), "get_font" in globals(), "TEAM_RGB" in globals(), "draw_base_layer" in globals(), "smooth_boxes" in globals())

### 12b. Carrier check on real video frames
For the cases in `CARRIER_CASES`: proposes the carrier from the tracking (nearest player to the ball), prints a half-second table, and shows a contact sheet of real frames with the suggested carrier marked. Confirm by eye, then put the correct lines into `CARRIER_SEGMENTS` (section 2b) and re-run from 2b.

In [ ]:
from IPython.display import display, Image as IPImage
CARRIER_CASES = ["b010_ATM_03-22", "b076_BAR_21-20", "b099_ATM_27-43", "b105_ATM_28-46",
                 "b113_BAR_31-59", "b121_ATM_34-46", "b133_ATM_37-26", "b137_BAR_38-24"]
CARRIER_MAX_DIST_M = 2.5        # nearest player to the ball must be this close to be proposed as the carrier
CARRIER_MIN_RUN_S = 0.3         # a proposed carrier must stay the nearest player this long
SHEET_FRAMES = 6                # frames shown per case (2 columns x 3 rows)
SHEET_DIR = OUT_DIR / "carrier_check"; SHEET_DIR.mkdir(exist_ok=True)
_SHEET_COLS = ["frame_idx", "track_id", "role", "team", "number", "x1", "y1", "x2", "y2", "foot_px_x", "foot_px_y"]


def nearest_to_ball(f0, f1):
    """For every frame with a usable ball position: the nearest tracked player (goalkeepers included) and his distance in metres."""
    b = ball_raw[(ball_raw.frame_idx >= f0) & (ball_raw.frame_idx <= f1) & ball_raw.ball_x.notna()][["frame_idx", "ball_x", "ball_y"]]
    pl = player[(player.frame_idx >= f0) & (player.frame_idx <= f1)][["frame_idx", "track_id", "team", "x", "y"]]
    m = pl.merge(b, on="frame_idx")
    m["dist"] = np.hypot(m.x - m.ball_x, m.y - m.ball_y)
    return m.loc[m.groupby("frame_idx")["dist"].idxmin()].set_index("frame_idx")


def carrier_segments_from(nn, frames_ok, max_dist=CARRIER_MAX_DIST_M, min_run=CARRIER_MIN_RUN_S):
    """Runs of consecutive frames where the same player is the nearest one (<= max_dist) -> [(f0, f1, track_id, mean_dist)]"""
    nn = nn[(nn.dist <= max_dist) & nn.index.isin(frames_ok)]
    segs, cur = [], None
    for f, r in nn.iterrows():
        if cur and int(r.track_id) == cur["tid"] and f - cur["f1"] <= 5:
            cur["f1"] = f; cur["d"].append(r.dist)
        else:
            if cur: segs.append(cur)
            cur = dict(tid=int(r.track_id), f0=f, f1=f, d=[r.dist])
    if cur: segs.append(cur)
    return [(s["f0"], s["f1"], s["tid"], float(np.mean(s["d"]))) for s in segs if (s["f1"] - s["f0"] + 1) >= min_run * FPS]


def carrier_review(name, t0=None, t1=None, only_missing=True, pre=3.0, post=6.0):
    info = parse_clip(name)
    row = buildups[(buildups.team_name == info["team_name"]) & (buildups.start_time == info["start_time"])]
    if row.empty:
        print(f"{name}: no build-up at that time"); return
    b = row.iloc[0]
    f0 = to_frame(t0) if t0 else int(b.end_frame - pre * FPS)
    f1 = to_frame(t1) if t1 else int(b.end_frame + post * FPS)
    win = ball_raw[(ball_raw.frame_idx >= f0) & (ball_raw.frame_idx <= f1)]
    missing = set(win.loc[win.carrier_track_id.isna(), "frame_idx"]) if only_missing else set(win.frame_idx)
    nn = nearest_to_ball(f0, f1)
    segs = carrier_segments_from(nn, missing)

    print(f"\n{'=' * 100}\n{name}: build-up {int(b.buildup_id)}, outcome {b.outcome}, ends {_clock(b.end_frame)}; window {_clock(f0)} -> {_clock(f1)}"
          f"  ({'only frames without a carrier' if only_missing else 'all frames'})")
    print(f"{'clock':>8}  {'carrier in table':<32} {'nearest player to the ball':<36} dist")
    for f in range(f0, f1 + 1, int(0.5 * FPS)):
        tc = win.loc[win.frame_idx == f, "carrier_track_id"]
        tc = tc.iloc[0] if len(tc) else np.nan
        n_ = f"{_who(nn.at[f, 'track_id'])}" if f in nn.index else "- (no ball position)"
        d_ = f"{nn.at[f, 'dist']:.1f} m" if f in nn.index else ""
        print(f"{_clock(f):>8}  {_who(tc):<32} {n_:<36} {d_}")
    print("\nsuggested segments (paste the correct ones into CARRIER_SEGMENTS):")
    if not segs:
        print("  none: the ball is not within %.1f m of any player (long ball / ball in the air / no ball position)." % CARRIER_MAX_DIST_M)
    for a, z, tid, d in segs:
        print(f'    ("{_clock(a)}", "{_clock(z)}", {tid}),   # {_who(tid)}, mean distance {d:.1f} m')

    # ---- contact sheet of REAL video frames, suggested carrier marked with the white triangle
    sug = {}
    for a, z, tid, d in segs:
        for f in range(a, z + 1):
            sug[f] = tid
    pick = sorted(set(int(x) for x in np.linspace(f0, f1, SHEET_FRAMES)))
    try:
        pl_all = pd.read_parquet(PLAYER_PATH, columns=_SHEET_COLS, filters=[("frame_idx", ">=", f0), ("frame_idx", "<=", f1)])
        bpx = pd.read_parquet(BALL_PATH, columns=["frame_idx", "ball_px_x", "ball_px_y"], filters=[("frame_idx", ">=", f0), ("frame_idx", "<=", f1)]).set_index("frame_idx")
    except Exception:
        pl_all = pd.read_parquet(PLAYER_PATH, columns=_SHEET_COLS); pl_all = pl_all[(pl_all.frame_idx >= f0) & (pl_all.frame_idx <= f1)]
        bpx = pd.read_parquet(BALL_PATH, columns=["frame_idx", "ball_px_x", "ball_px_y"]).set_index("frame_idx")
    cap = cv2.VideoCapture(str(VIDEO_PATH)); tiles = []
    for f in pick:
        cap.set(cv2.CAP_PROP_POS_FRAMES, f + FRAME_OFFSET); ok_, img = cap.read()
        if not ok_:
            continue
        pf_ = pl_all[pl_all.frame_idx == f]
        brow = bpx.loc[f] if f in bpx.index else None
        bxy = (brow.ball_px_x, brow.ball_px_y) if brow is not None and pd.notna(brow.ball_px_x) else None
        draw_base_layer(img, pf_, bxy, sug.get(f))
        tc = win.loc[win.frame_idx == f, "carrier_track_id"]; tc = tc.iloc[0] if len(tc) else np.nan
        cap_txt = f"{_clock(f)}  table: {_who(tc)}  |  suggested: {_who(sug[f]) if f in sug else '-'}"
        img = cv2.resize(img, (960, 540))
        cv2.rectangle(img, (0, 0), (960, 30), (0, 0, 0), -1)
        cv2.putText(img, cap_txt, (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
        tiles.append(img)
    cap.release()
    if tiles:
        if len(tiles) % 2: tiles.append(np.zeros_like(tiles[0]))
        sheet = np.vstack([np.hstack(tiles[i:i + 2]) for i in range(0, len(tiles), 2)])
        out = SHEET_DIR / f"{Path(name).stem}.jpg"; cv2.imwrite(str(out), sheet)
        ok_, buf = cv2.imencode(".jpg", sheet, [cv2.IMWRITE_JPEG_QUALITY, 85]); display(IPImage(data=buf.tobytes(), width=1300))
        print("sheet saved to", out)


for n in CARRIER_CASES:
    carrier_review(n)


## 13. Camera calibration & ground graphics
Rebuilds the pitch→pixel homography per frame so thirds, press radius, defensive line and passing lanes can be drawn on the grass. This cell is a preview/check on one build-up.

In [ ]:
# ================================================================== camera calibration (pitch metres -> pixels) + ground graphics
# The tracking tables contain, for every tracked player and frame, the feet position in the video (foot_px_x/y) AND on the pitch
# (pitch_x_raw / pitch_y_raw). Those pairs let us rebuild the camera mapping of every frame (a homography, fitted with RANSAC), so graphics
# can be drawn ON the grass with correct perspective. Everything on the ground is drawn BEHIND the players (player boxes cut out of it).
# needs: the annotation cell v5 (draw_base_layer, get_font), Cell 12b (POS_OF, NUM_OF) and Cell 9 (buildups_p, pr_i, loc_i)
from PIL import Image, ImageDraw

LAYERS = dict(thirds=True, press_radius=True, defensive_line=True, pass_lanes=True)       # switch layers on / off
CALIB_CHECK_LINES = True        # preview only: draw the projected pitch lines (cyan) so you can compare them with the real lines on the grass
RING_OUTER_M, RING_INNER_M = P["PRESS_DIST_M"], P["PRESS_TIGHT_M"]
LANE_BLOCK_M, LANE_CONTEST_M = 1.5, 3.0     # opponent closer than this to a passing lane: blocked / contested
LANE_MIN_M, LANE_MAX_M = 3.0, 45.0          # lanes shorter / longer than this are not drawn
INCLUDE_GK_LANES = False
MASK_SHRINK, MASK_FOOT_CUT = 0.15, 0.08    # player box used to hide ground graphics: shrunk sideways, and the feet are left free
CALIB_COLS = ["frame_idx", "track_id", "role", "team", "number", "x1", "y1", "x2", "y2", "foot_px_x", "foot_px_y",
              "pitch_x_raw", "pitch_y_raw", "pitch_valid", "source"]
LANE_RGB = dict(open=(70, 210, 90), contested=(255, 190, 30), blocked=(235, 60, 60))

# ------------------------------------------------------------------ homography per frame
def fit_homographies(df, thr_px=6.0, min_pts=6):
    d_ = df[df["pitch_valid"].fillna(False).astype(bool) & df["pitch_x_raw"].notna() & df["foot_px_x"].notna()]
    out = {}
    for f, g in d_.groupby("frame_idx"):
        if len(g) < min_pts:
            continue
        src = g[["pitch_x_raw", "pitch_y_raw"]].to_numpy(np.float64); dst = g[["foot_px_x", "foot_px_y"]].to_numpy(np.float64)
        H, _ = cv2.findHomography(src, dst, cv2.RANSAC, thr_px)
        if H is None:
            continue
        err = np.hypot(*(cv2.perspectiveTransform(src.reshape(-1, 1, 2), H).reshape(-1, 2) - dst).T)
        ok = err < thr_px
        if ok.sum() < min_pts - 1:
            continue
        out[int(f)] = dict(H=H / H[2, 2], err=float(np.median(err[ok])), n=int(ok.sum()))
    return out

def smooth_homographies(raw, frames, max_gap=12, win=5):
    """Fill short gaps and lightly smooth the 9 matrix entries over time (centred, so no lag)."""
    A = np.full((len(frames), 9), np.nan)
    for i, f in enumerate(frames):
        if f in raw:
            A[i] = raw[f]["H"].ravel()
    A = pd.DataFrame(A).interpolate(limit=max_gap, limit_area="inside").to_numpy().copy()
    S = pd.DataFrame(A).rolling(win, center=True, min_periods=1).mean().to_numpy().copy()
    S[np.isnan(A).any(axis=1)] = np.nan
    return {int(f): (S[i].reshape(3, 3) if not np.isnan(S[i]).any() else None) for i, f in enumerate(frames)}

def proj(H, xy):
    """pitch metres (N,2) -> pixels (N,2); NaN where the point is behind the camera."""
    xy = np.asarray(xy, float).reshape(-1, 2)
    p = np.c_[xy, np.ones(len(xy))] @ H.T
    w = p[:, 2]
    with np.errstate(divide="ignore", invalid="ignore"):
        out = p[:, :2] / w[:, None]
    out[w <= 1e-9] = np.nan
    return out

def clip_poly(pts, W, H, m=60):
    """Sutherland-Hodgman clip of a polygon to the image (plus a margin), so far-away corners cannot distort it."""
    x0, y0, x1, y1 = -m, -m, W + m, H + m
    def step(poly, inside, inter):
        out = []
        for i in range(len(poly)):
            a, b = poly[i - 1], poly[i]
            ia, ib = inside(a), inside(b)
            if ia and ib:
                out.append(b)
            elif ia and not ib:
                out.append(inter(a, b))
            elif (not ia) and ib:
                out.append(inter(a, b)); out.append(b)
        return out
    poly = [tuple(map(float, p)) for p in pts]
    for inside, inter in (
        (lambda p: p[0] >= x0, lambda a, b: (x0, a[1] + (b[1] - a[1]) * (x0 - a[0]) / (b[0] - a[0]))),
        (lambda p: p[0] <= x1, lambda a, b: (x1, a[1] + (b[1] - a[1]) * (x1 - a[0]) / (b[0] - a[0]))),
        (lambda p: p[1] >= y0, lambda a, b: (a[0] + (b[0] - a[0]) * (y0 - a[1]) / (b[1] - a[1]), y0)),
        (lambda p: p[1] <= y1, lambda a, b: (a[0] + (b[0] - a[0]) * (y1 - a[1]) / (b[1] - a[1]), y1))):
        if not poly:
            break
        poly = step(poly, inside, inter)
    return poly

# ------------------------------------------------------------------ passing lanes (pitch geometry decides open / contested / blocked)
def compute_lanes(carrier_id, pos_f, foot):
    """Lane from the carrier to each teammate. blocked: an opponent within LANE_BLOCK_M of the lane; contested: within LANE_CONTEST_M."""
    if carrier_id is None or carrier_id not in foot:
        return []
    car = pos_f[pos_f.track_id == carrier_id]
    if len(car) == 0:
        return []
    car = car.iloc[0]; cxy = np.array([car.x, car.y]); team = int(car.team)
    mates = pos_f[(pos_f.team == team) & (pos_f.track_id != carrier_id)]
    if not INCLUDE_GK_LANES:
        mates = mates[mates.role == "player"]
    opps = pos_f[pos_f.team != team][["x", "y"]].to_numpy()
    lanes = []
    for r in mates.itertuples():
        if r.track_id not in foot:
            continue
        ab = np.array([r.x, r.y]) - cxy; L = float(np.hypot(*ab))
        if L < LANE_MIN_M or L > LANE_MAX_M:
            continue
        status, dmin = "open", np.inf
        if len(opps):
            t = ((opps - cxy) @ ab) / (L * L)
            valid = (t > 0.1) & (t < 0.9)                                   # opponents beside / behind either end do not block
            if valid.any():
                closest = cxy + np.clip(t, 0, 1)[:, None] * ab
                dmin = float(np.hypot(*(opps - closest).T)[valid].min())
                status = "blocked" if dmin < LANE_BLOCK_M else ("contested" if dmin < LANE_CONTEST_M else "open")
        lanes.append((foot[carrier_id], foot[r.track_id], status, int(r.track_id), L, dmin))
    return lanes

# ------------------------------------------------------------------ ground layer: drawn on the grass, hidden behind players
def ground_layer(img, H, g, boxes_f):
    """g: team (build-up team), target_m (x, y) or None, n_press, def_team, def_line_x or None, lanes, check_lines."""
    Hh, Ww = img.shape[:2]; s = Hh / 1080
    base = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).convert("RGBA")
    ov = Image.new("RGBA", base.size, (0, 0, 0, 0)); dr = ImageDraw.Draw(ov)
    ap = TEAM_ATTACKS_POSITIVE_X[g["team"]]

    def seg_m(p0, p1, rgba, width):
        q = proj(H, [p0, p1])
        if np.isnan(q).any():
            return None
        c = lambda v: tuple(int(t) for t in np.clip(v, -1e7, 1e7))
        ok, a, b = cv2.clipLine((0, 0, Ww, Hh), c(q[0]), c(q[1]))
        if ok:
            dr.line([a, b], fill=rgba, width=width)
            return a, b
        return None

    def rect_m(x0, x1, rgba):
        p = proj(H, [(x0, 0), (x1, 0), (x1, PITCH_WIDTH), (x0, PITCH_WIDTH)])
        if not np.isnan(p).any():
            poly = clip_poly(p, Ww, Hh)
            if len(poly) >= 3:
                dr.polygon(poly, fill=rgba)

    def ring_m(cx, cy, R, fill, outline, width):
        a = np.linspace(0, 2 * np.pi, 73)[:-1]
        p = proj(H, np.c_[cx + R * np.cos(a), cy + R * np.sin(a)])
        if np.isnan(p).any():
            return
        pts = [tuple(map(float, q)) for q in p]
        if fill:
            dr.polygon(pts, fill=fill)
        dr.line(pts + [pts[0]], fill=outline, width=width)

    L = g["layers"]; label_at = None
    if L.get("thirds"):                                                     # thirds in the build-up team's attacking direction
        own, mid, att = ((0, 35), (35, 70), (70, 105)) if ap else ((70, 105), (35, 70), (0, 35))
        rect_m(*own, (0, 0, 0, 30)); rect_m(*att, (140, 255, 140, 34))
        x_entry, x_other = (70, 35) if ap else (35, 70)
        seg_m((x_other, 0), (x_other, PITCH_WIDTH), (255, 255, 255, 120), max(2, int(2 * s)))
        seg_m((x_entry, 0), (x_entry, PITCH_WIDTH), TEAM_RGB[g["team"]] + (215,), max(3, int(4 * s)))     # entry line of the final third
    if L.get("press_radius") and g["target_m"] is not None:                 # real pressing radius on the grass
        n = int(g["n_press"]); col = (255, 255, 255) if n == 0 else ((255, 150, 30) if n == 1 else (235, 50, 50))
        ring_m(*g["target_m"], RING_OUTER_M, col + (26,), col + (205,), max(2, int(3 * s)))
        ring_m(*g["target_m"], RING_INNER_M, None, col + (150,), max(1, int(2 * s)))
    if L.get("defensive_line") and g["def_line_x"] is not None:             # the defending team's last outfield player, across the pitch
        col = TEAM_RGB[g["def_team"]]
        seg_m((g["def_line_x"], 0), (g["def_line_x"], PITCH_WIDTH), col + (225,), max(3, int(4 * s)))
        ys = np.linspace(0, PITCH_WIDTH, 60); q = proj(H, np.c_[np.full(60, g["def_line_x"]), ys])
        vis = [(x, y) for x, y in q if not np.isnan(x) and 160 < x < Ww - 160 and 120 < y < Hh - 120]
        if vis:
            label_at = (vis[len(vis) // 2], col, f"{TEAM_NAMES[g['def_team']]} last defender")
    if L.get("pass_lanes"):
        for p1, p2, status, *_ in sorted(g["lanes"], key=lambda l: {"open": 0, "contested": 1, "blocked": 2}[l[2]], reverse=True):
            dr.line([(float(p1[0]), float(p1[1])), (float(p2[0]), float(p2[1]))], fill=LANE_RGB[status] + (205,),
                    width=max(1, int((1 if status == "blocked" else 2) * s)))
    if g.get("check_lines"):                                                # projected pitch lines, only to verify the calibration
        cyan = (0, 255, 255, 230)
        for p0, p1 in (((0, 0), (105, 0)), ((105, 0), (105, 68)), ((105, 68), (0, 68)), ((0, 68), (0, 0)), ((52.5, 0), (52.5, 68)),
                       ((0, 13.84), (16.5, 13.84)), ((16.5, 13.84), (16.5, 54.16)), ((16.5, 54.16), (0, 54.16)),
                       ((105, 13.84), (88.5, 13.84)), ((88.5, 13.84), (88.5, 54.16)), ((88.5, 54.16), (105, 54.16))):
            seg_m(p0, p1, cyan, 2)
        a = np.linspace(0, 2 * np.pi, 73)[:-1]; q = proj(H, np.c_[52.5 + 9.15 * np.cos(a), 34 + 9.15 * np.sin(a)])
        if not np.isnan(q).any():
            pts = [tuple(map(float, t)) for t in q]; dr.line(pts + [pts[0]], fill=cyan, width=2)

    arr = np.array(ov)                                                      # hide everything behind the players (soft edges)
    mask = np.zeros(arr.shape[:2], np.float32)
    for r in boxes_f.itertuples():
        if pd.isna(r.x1):
            continue
        w = r.x2 - r.x1
        xa, xb = int(max(r.x1 + w * MASK_SHRINK, 0)), int(max(r.x2 - w * MASK_SHRINK, 0))
        ya, yb = int(max(r.y1, 0)), int(max(r.y2 - (r.y2 - r.y1) * MASK_FOOT_CUT, 0))
        mask[ya:yb, xa:xb] = 1.0
    mask = np.clip(cv2.GaussianBlur(mask, (0, 0), max(2.0, 3.5 * s)) * 1.6, 0, 1)
    arr[..., 3] = (arr[..., 3].astype(np.float32) * (1 - mask)).astype(np.uint8)
    out = Image.alpha_composite(base, Image.fromarray(arr, "RGBA"))
    if label_at is not None:                                                # the label sits on top of everything
        (x, y), col, txt = label_at; d2 = ImageDraw.Draw(out); f_ = get_font(max(11, int(13 * s)), True); w_ = d2.textlength(txt, font=f_)
        d2.rounded_rectangle([x - w_ / 2 - 6 * s, y - 12 * s, x + w_ / 2 + 6 * s, y + 10 * s], radius=int(5 * s), fill=col + (225,))
        d2.text((x, y - int(1 * s)), txt, font=f_, fill=(255, 255, 255, 255), anchor="mm")
    img[:] = cv2.cvtColor(np.array(out.convert("RGB")), cv2.COLOR_RGB2BGR)

def draw_lane_legend(img, lanes, carrier_id):
    """Small legend: how many lanes from the carrier are open / contested / blocked."""
    H_, W_ = img.shape[:2]; s = H_ / 1080
    base = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).convert("RGBA")
    ov = Image.new("RGBA", base.size, (0, 0, 0, 0)); dr = ImageDraw.Draw(ov)
    f1, f2 = get_font(max(11, int(14 * s)), True), get_font(max(10, int(13 * s)), False)
    who = f"#{NUM_OF.get(int(carrier_id), '?')} {POS_OF.get(int(carrier_id), '')}" if carrier_id is not None else "carrier"
    x0, y0 = int(16 * s), int(78 * s)
    dr.rounded_rectangle([x0, y0, x0 + int(300 * s), y0 + int(30 * s)], radius=int(7 * s), fill=(18, 18, 18, 150))
    dr.text((x0 + int(10 * s), y0 + int(15 * s)), f"pass lanes from {who}", font=f1, fill=(255, 255, 255, 255), anchor="lm")
    x = x0 + int(10 * s) + dr.textlength(f"pass lanes from {who}", font=f1) + int(14 * s)
    for k in ("open", "contested", "blocked"):
        n = sum(1 for l in lanes if l[2] == k)
        dr.ellipse([x, y0 + int(10 * s), x + int(10 * s), y0 + int(20 * s)], fill=LANE_RGB[k] + (255,))
        dr.text((x + int(15 * s), y0 + int(15 * s)), str(n), font=f2, fill=(255, 255, 255, 255), anchor="lm"); x += int(34 * s)
    img[:] = cv2.cvtColor(np.array(Image.alpha_composite(base, ov).convert("RGB")), cv2.COLOR_RGB2BGR)

# ================================================================== preview on real frames: calibration quality + all ground layers
cap = cv2.VideoCapture(str(VIDEO_PATH)); assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
cand = buildups_p[(buildups_p.outcome == "lost") & (buildups_p.pressed_at_end == True) &
                  (buildups_p.target_coverage > 0.8) & (buildups_p.ball_target_share < 0.3) & (buildups_p.duration_sec > 4)]
b = cand.iloc[len(cand) // 2]
team = int(b.team); first = int(b.first_press_frame) if pd.notna(b.first_press_frame) else int(b.start_frame)
test_frames = [first, int((first + b.end_frame) // 2), int(b.end_frame)]
fr_win = np.arange(int(b.start_frame), int(b.end_frame) + 1)

try:
    calib = pd.read_parquet(PLAYER_PATH, columns=CALIB_COLS, filters=[("frame_idx", ">=", int(fr_win[0])), ("frame_idx", "<=", int(fr_win[-1]))])
except Exception:
    calib = pd.read_parquet(PLAYER_PATH, columns=CALIB_COLS); calib = calib[(calib.frame_idx >= fr_win[0]) & (calib.frame_idx <= fr_win[-1])]
Hraw = fit_homographies(calib)
Hs = smooth_homographies(Hraw, fr_win)
errs = np.array([v["err"] for v in Hraw.values()]) if Hraw else np.array([np.nan])
print(f"build-up {int(b.buildup_id)} ({b.team_name}, {b.start_time}): homography fitted for {len(Hraw)} of {len(fr_win)} frames "
      f"({len(Hraw) / len(fr_win) * 100:.0f}%), usable after gap filling: {sum(v is not None for v in Hs.values())}")
print(f"reprojection error of the players' feet (px): median {np.nanmedian(errs):.2f}, 95th percentile {np.nanpercentile(errs, 95):.2f}; "
      f"inlier players per frame: median {np.median([v['n'] for v in Hraw.values()]) if Hraw else 0:.0f}")

bpx = pd.read_parquet(BALL_PATH, columns=["frame_idx", "ball_px_x", "ball_px_y", "carrier_track_id"])
bpx = bpx[bpx.frame_idx.isin(test_frames)].set_index("frame_idx")
shots = []
for f in test_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, f + FRAME_OFFSET); ok_, img = cap.read()
    if not ok_:
        continue
    bf = calib[calib.frame_idx == f]; H = Hs.get(f)
    row = bpx.loc[f]; carrier_id = row.carrier_track_id if pd.notna(row.carrier_track_id) else None
    ball_xy = (row.ball_px_x, row.ball_px_y) if pd.notna(row.ball_px_x) else None
    pos_f = player[player.frame_idx == f]
    foot = {int(r.track_id): (r.foot_px_x, r.foot_px_y) for r in bf.itertuples() if pd.notna(r.foot_px_x)}
    lanes = compute_lanes(carrier_id, pos_f, foot)
    if H is not None:
        tgt = None
        if carrier_id is not None:
            cr = bf[bf.track_id == carrier_id]
            if len(cr) and pd.notna(cr.iloc[0].pitch_x_raw):
                tgt = (float(cr.iloc[0].pitch_x_raw), float(cr.iloc[0].pitch_y_raw))
        if tgt is None and f in loc_i.index and pd.notna(loc_i.loc[f, "x"]):
            tgt = (float(loc_i.loc[f, "x"]), float(loc_i.loc[f, "y"]))
        deff = pos_f[(pos_f.team == 1 - team) & (pos_f.role == "player")]
        line_x = (float(deff.x.min()) if TEAM_ATTACKS_POSITIVE_X[1 - team] else float(deff.x.max())) if len(deff) else None
        g = dict(team=team, target_m=tgt, n_press=int(pr_i.loc[f, "n_pressing"]) if f in pr_i.index else 0, def_team=1 - team,
                 def_line_x=line_x, lanes=lanes, check_lines=CALIB_CHECK_LINES, layers=LAYERS)
        ground_layer(img, H, g, bf)
    else:
        print(f"frame {f}: no homography, ground graphics skipped")
    draw_base_layer(img, bf, ball_xy, carrier_id)
    draw_lane_legend(img, lanes, carrier_id)
    path = OUT_DIR / f"ground_check_{f}.png"; cv2.imwrite(str(path), img); shots.append((f, img))
    print(f"frame {f} ({mmss(f)}): lanes open/contested/blocked = "
          f"{sum(l[2] == 'open' for l in lanes)}/{sum(l[2] == 'contested' for l in lanes)}/{sum(l[2] == 'blocked' for l in lanes)}")
cap.release()
print("saved full-resolution check frames to", OUT_DIR)
if shots:
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(shots[0][1], cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title(f"frame {shots[0][0]} ({mmss(shots[0][0])}): cyan = projected pitch lines, compare with the real lines"); plt.tight_layout(); plt.show()

## 14. Final clip rendering
Renders every selected clip (lost under pressure, kept under pressure, lost without pressure). Clips already on disk are **skipped** unless `OVERWRITE = True`.

In [ ]:
# ================================================================== settings
# FINAL CLIP CELL. needs: Cell 12 (video + style), Cell 12b (POS_OF, NUM_OF), the annotation cell v5, and the analysis cells (buildups_p, pr_i, loc_i, pf, player, ball)
from PIL import Image, ImageDraw, ImageFont
_needed = ["POS_OF", "NUM_OF", "draw_base_layer", "smooth_boxes", "get_font", "TEAM_RGB", "STYLE", "VIDEO_PATH", "FRAME_OFFSET",
           "ALLOWED_PLAYER_SOURCES", "ALLOWED_BALL_SOURCES", "buildups_p", "pr_i", "loc_i", "pf", "player", "ball", "TEAM_ATTACKS_POSITIVE_X"]
_missing = [n for n in _needed if n not in globals()]
assert not _missing, f"run the earlier cells first, these are missing: {_missing}"
CLIP_DIR = OUT_DIR / "clips3"; CLIP_DIR.mkdir(exist_ok=True)       # one sub-folder per category is created below
PRE_SEC, POST_SEC, MAX_CLIP_SEC, FREEZE_END_SEC = 2.0, 2.0, 20.0, 1.0
SLOWDOWN = 1            # 1 = real time, 2 = every frame shown twice (half speed)
CODEC = "mp4v"          # plays in VLC / Windows Media Player; use "avc1" (H.264) only if your OpenCV has it
N_PER_CATEGORY = None      # clips per category (3 categories); lower it if rendering takes too long
MAX_PER_TEAM = 10**9        # at most this many clips of one team inside a category
PAIR_MAX_DIST_M = 15.0  # draw a pair line only if the defender and his nearest opponent are closer than this (None = always)
PAIR_LABEL_MAX_M = 5.0  # small distance label on the line only for pairs closer than this (0 = no labels)
TEAM_TXT = {t: str(TEAM_NAMES[t]) for t in (0, 1)}
MM_W, MM_H = 360, 235   # minimap size in pixels (at 1080p)
bgr2rgb = lambda c: (c[2], c[1], c[0])
COL = dict(press=(0, 140, 255), hot=(0, 0, 255), good=(70, 190, 60), bad=(50, 50, 235), neutral=(150, 150, 150))   # BGR


# ground graphics (drawn on the grass with the per-frame camera mapping, always BEHIND the players)
LAYERS = dict(thirds=True, press_radius=True, defensive_line=True, pass_lanes=True, pair_lines=True)    # switch layers on / off
RING_OUTER_M, RING_INNER_M = P["PRESS_DIST_M"], P["PRESS_TIGHT_M"]
LANE_BLOCK_M, LANE_CONTEST_M = 1.5, 3.0     # opponent closer than this to a passing lane: blocked / contested
LANE_MIN_M, LANE_MAX_M = 3.0, 45.0          # lanes shorter / longer than this are not drawn
INCLUDE_GK_LANES = False
MAX_MEDIAN_ERR_PX = 5.0                     # a frame whose camera fit is worse than this gets no thirds / ring / line (lanes and pair lines still work)
MASK_SHRINK, MASK_FOOT_CUT = 0.15, 0.08     # player box used to hide ground graphics: shrunk sideways, the feet are left free
CALIB_COLS = ["frame_idx", "track_id", "role", "team", "number", "x1", "y1", "x2", "y2", "foot_px_x", "foot_px_y",
              "pitch_x_raw", "pitch_y_raw", "pitch_valid", "source"]
LANE_RGB = dict(open=(70, 210, 90), contested=(255, 190, 30), blocked=(235, 60, 60))

# ================================================================== which clips
bp = buildups_p.copy()
bp["score"] = bp["max_pressers"].fillna(0) * 10 + bp["pressure_share"].fillna(0) * 10
base = bp[(bp.target_coverage > 0.8) & (bp.ball_target_share < 0.5) & (bp.duration_sec.between(3, 25))]
categories = {
    "lost_under_pressure":   ("BALL LOST UNDER PRESSURE", base[(base.outcome == "lost") & (base.pressed_at_end == True)]),
    "kept_under_pressure":   ("PROGRESSED DESPITE PRESSURE", base[(base.outcome == "success") & (base.max_pressers >= 2)]),
    "lost_without_pressure": ("BALL LOST WITHOUT PRESSURE (contrast)", base[(base.outcome == "lost") & (base.pressed_at_end == False)]),
}

def pick(df, n):
    out, per_team = [], {}
    for r in df.sort_values("score", ascending=False).itertuples():
        if per_team.get(r.team_name, 0) >= MAX_PER_TEAM:
            continue
        out.append(r.buildup_id); per_team[r.team_name] = per_team.get(r.team_name, 0) + 1
        if len(out) == n:
            break
    return out

selected = [(cat, title, bid) for cat, (title, df) in categories.items() for bid in pick(df, N_PER_CATEGORY)]
print("candidate build-ups available per category:", {c: len(df) for c, (t, df) in categories.items()})
print(f"clips to render: {len(selected)}")
for cat, title, bid in selected:
    r = buildups_p[buildups_p.buildup_id == bid].iloc[0]
    print(f"  {cat:22s} build-up {bid:3d} {r.team_name} {r.start_time} dur {r.duration_sec:4.1f}s  max pressers {int(r.max_pressers)}  outcome {r.outcome}")

# ------------------------------------------------------------------ homography per frame
def fit_homographies(df, thr_px=6.0, min_pts=6):
    d_ = df[df["pitch_valid"].fillna(False).astype(bool) & df["pitch_x_raw"].notna() & df["foot_px_x"].notna()]
    out = {}
    for f, g in d_.groupby("frame_idx"):
        if len(g) < min_pts:
            continue
        src = g[["pitch_x_raw", "pitch_y_raw"]].to_numpy(np.float64); dst = g[["foot_px_x", "foot_px_y"]].to_numpy(np.float64)
        H, _ = cv2.findHomography(src, dst, cv2.RANSAC, thr_px)
        if H is None:
            continue
        err = np.hypot(*(cv2.perspectiveTransform(src.reshape(-1, 1, 2), H).reshape(-1, 2) - dst).T)
        ok = err < thr_px
        if ok.sum() < min_pts - 1 or np.median(err[ok]) > MAX_MEDIAN_ERR_PX:
            continue
        out[int(f)] = dict(H=H / H[2, 2], err=float(np.median(err[ok])), n=int(ok.sum()))
    return out

def smooth_homographies(raw, frames, max_gap=12, win=5):
    """Fill short gaps and lightly smooth the 9 matrix entries over time (centred, so no lag)."""
    A = np.full((len(frames), 9), np.nan)
    for i, f in enumerate(frames):
        if f in raw:
            A[i] = raw[f]["H"].ravel()
    A = pd.DataFrame(A).interpolate(limit=max_gap, limit_area="inside").to_numpy().copy()
    S = pd.DataFrame(A).rolling(win, center=True, min_periods=1).mean().to_numpy().copy()
    S[np.isnan(A).any(axis=1)] = np.nan
    return {int(f): (S[i].reshape(3, 3) if not np.isnan(S[i]).any() else None) for i, f in enumerate(frames)}

def proj(H, xy):
    """pitch metres (N,2) -> pixels (N,2); NaN where the point is behind the camera."""
    xy = np.asarray(xy, float).reshape(-1, 2)
    p = np.c_[xy, np.ones(len(xy))] @ H.T
    w = p[:, 2]
    with np.errstate(divide="ignore", invalid="ignore"):
        out = p[:, :2] / w[:, None]
    out[w <= 1e-9] = np.nan
    return out

def clip_poly(pts, W, H, m=60):
    """Sutherland-Hodgman clip of a polygon to the image (plus a margin), so far-away corners cannot distort it."""
    x0, y0, x1, y1 = -m, -m, W + m, H + m
    def step(poly, inside, inter):
        out = []
        for i in range(len(poly)):
            a, b = poly[i - 1], poly[i]
            ia, ib = inside(a), inside(b)
            if ia and ib:
                out.append(b)
            elif ia and not ib:
                out.append(inter(a, b))
            elif (not ia) and ib:
                out.append(inter(a, b)); out.append(b)
        return out
    poly = [tuple(map(float, p)) for p in pts]
    for inside, inter in (
        (lambda p: p[0] >= x0, lambda a, b: (x0, a[1] + (b[1] - a[1]) * (x0 - a[0]) / (b[0] - a[0]))),
        (lambda p: p[0] <= x1, lambda a, b: (x1, a[1] + (b[1] - a[1]) * (x1 - a[0]) / (b[0] - a[0]))),
        (lambda p: p[1] >= y0, lambda a, b: (a[0] + (b[0] - a[0]) * (y0 - a[1]) / (b[1] - a[1]), y0)),
        (lambda p: p[1] <= y1, lambda a, b: (a[0] + (b[0] - a[0]) * (y1 - a[1]) / (b[1] - a[1]), y1))):
        if not poly:
            break
        poly = step(poly, inside, inter)
    return poly

# ------------------------------------------------------------------ passing lanes (pitch geometry decides open / contested / blocked)
def compute_lanes(carrier_id, pos_f, foot):
    """Lane from the carrier to each teammate. blocked: an opponent within LANE_BLOCK_M of the lane; contested: within LANE_CONTEST_M."""
    if carrier_id is None or carrier_id not in foot:
        return []
    car = pos_f[pos_f.track_id == carrier_id]
    if len(car) == 0:
        return []
    car = car.iloc[0]; cxy = np.array([car.x, car.y]); team = int(car.team)
    mates = pos_f[(pos_f.team == team) & (pos_f.track_id != carrier_id)]
    if not INCLUDE_GK_LANES:
        mates = mates[mates.role == "player"]
    opps = pos_f[pos_f.team != team][["x", "y"]].to_numpy()
    lanes = []
    for r in mates.itertuples():
        if r.track_id not in foot:
            continue
        ab = np.array([r.x, r.y]) - cxy; L = float(np.hypot(*ab))
        if L < LANE_MIN_M or L > LANE_MAX_M:
            continue
        status, dmin = "open", np.inf
        if len(opps):
            t = ((opps - cxy) @ ab) / (L * L)
            valid = (t > 0.1) & (t < 0.9)                                   # opponents beside / behind either end do not block
            if valid.any():
                closest = cxy + np.clip(t, 0, 1)[:, None] * ab
                dmin = float(np.hypot(*(opps - closest).T)[valid].min())
                status = "blocked" if dmin < LANE_BLOCK_M else ("contested" if dmin < LANE_CONTEST_M else "open")
        lanes.append((foot[carrier_id], foot[r.track_id], status, int(r.track_id), L, dmin))
    return lanes

# ------------------------------------------------------------------ ground layer: on the grass, hidden behind players
def pair_color(dist):
    return (0, 0, 255) if dist <= P["PRESS_TIGHT_M"] else ((0, 140, 255) if dist <= P["PRESS_DIST_M"] else (215, 215, 215))   # BGR

def ground_layer(img, H, g, boxes_f):
    """g: team, target_m, n_press, def_team, def_line_x, lanes, pairs_px.  H may be None: then only pixel-based lines (pairs, lanes) are drawn."""
    Hh, Ww = img.shape[:2]; s = Hh / 1080
    base = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).convert("RGBA")
    ov = Image.new("RGBA", base.size, (0, 0, 0, 0)); dr = ImageDraw.Draw(ov)
    ap = TEAM_ATTACKS_POSITIVE_X[g["team"]]
    L = g["layers"]; label_at = None

    def seg_m(p0, p1, rgba, width):
        q = proj(H, [p0, p1])
        if np.isnan(q).any():
            return
        c = lambda v: tuple(int(t) for t in np.clip(v, -1e7, 1e7))
        ok, a, b = cv2.clipLine((0, 0, Ww, Hh), c(q[0]), c(q[1]))
        if ok:
            dr.line([a, b], fill=rgba, width=width)

    def rect_m(x0, x1, rgba):
        p = proj(H, [(x0, 0), (x1, 0), (x1, PITCH_WIDTH), (x0, PITCH_WIDTH)])
        if not np.isnan(p).any():
            poly = clip_poly(p, Ww, Hh)
            if len(poly) >= 3:
                dr.polygon(poly, fill=rgba)

    def ring_m(cx, cy, R, fill, outline, width):
        a = np.linspace(0, 2 * np.pi, 73)[:-1]
        p = proj(H, np.c_[cx + R * np.cos(a), cy + R * np.sin(a)])
        if np.isnan(p).any():
            return
        pts = [tuple(map(float, q)) for q in p]
        if fill:
            dr.polygon(pts, fill=fill)
        dr.line(pts + [pts[0]], fill=outline, width=width)

    if H is not None:
        if L.get("thirds"):                                                 # only the final third: light tint + its entry line (own / middle third are not drawn)
            rect_m(*((70, 105) if ap else (0, 35)), (140, 255, 140, 34))
            x_entry = 70 if ap else 35
            seg_m((x_entry, 0), (x_entry, PITCH_WIDTH), TEAM_RGB[g["team"]] + (215,), max(3, int(4 * s)))   # entry line of the final third
        if L.get("press_radius") and g["target_m"] is not None:             # real pressing radius on the grass
            n = int(g["n_press"]); col = (255, 255, 255) if n == 0 else ((255, 150, 30) if n == 1 else (235, 50, 50))
            ring_m(*g["target_m"], RING_OUTER_M, col + (26,), col + (205,), max(2, int(3 * s)))
            ring_m(*g["target_m"], RING_INNER_M, None, col + (150,), max(1, int(2 * s)))
        if L.get("defensive_line") and g["def_line_x"] is not None:         # the defending team's last outfield player, across the pitch
            col = TEAM_RGB[g["def_team"]]
            seg_m((g["def_line_x"], 0), (g["def_line_x"], PITCH_WIDTH), col + (225,), max(3, int(4 * s)))
            ys = np.linspace(0, PITCH_WIDTH, 60); q = proj(H, np.c_[np.full(60, g["def_line_x"]), ys])
            vis = [(x, y) for x, y in q if not np.isnan(x) and 160 < x < Ww - 160 and 120 < y < Hh - 120]
            if vis:
                d_goal = g["def_line_x"] if TEAM_ATTACKS_POSITIVE_X[g["def_team"]] else PITCH_LENGTH - g["def_line_x"]   # distance to his own goal line
                label_at = (vis[len(vis) // 2], col, f"{TEAM_NAMES[g['def_team']]} last defender  ·  {d_goal:.1f} m from own goal")
    if L.get("pair_lines"):                                                 # each defender -> his nearest opponent (no goalkeepers)
        for p1, p2, dist in sorted(g["pairs_px"], key=lambda p: -p[2]):
            dr.line([(float(p1[0]), float(p1[1])), (float(p2[0]), float(p2[1]))], fill=pair_color(dist)[::-1] + (215,),
                    width=max(1, int((1 if dist > P["PRESS_DIST_M"] else 2) * s)))
    if L.get("pass_lanes"):                                                 # passing lanes from the carrier
        for p1, p2, status, *_ in sorted(g["lanes"], key=lambda l: {"open": 0, "contested": 1, "blocked": 2}[l[2]], reverse=True):
            dr.line([(float(p1[0]), float(p1[1])), (float(p2[0]), float(p2[1]))], fill=LANE_RGB[status] + (205,),
                    width=max(1, int((1 if status == "blocked" else 2) * s)))

    arr = np.array(ov)                                                      # hide everything behind the players (soft edges)
    mask = np.zeros(arr.shape[:2], np.float32)
    for r in boxes_f.itertuples():
        if pd.isna(r.x1):
            continue
        w = r.x2 - r.x1
        xa, xb = int(max(r.x1 + w * MASK_SHRINK, 0)), int(max(r.x2 - w * MASK_SHRINK, 0))
        ya, yb = int(max(r.y1, 0)), int(max(r.y2 - (r.y2 - r.y1) * MASK_FOOT_CUT, 0))
        mask[ya:yb, xa:xb] = 1.0
    mask = np.clip(cv2.GaussianBlur(mask, (0, 0), max(2.0, 3.5 * s)) * 1.6, 0, 1)
    arr[..., 3] = (arr[..., 3].astype(np.float32) * (1 - mask)).astype(np.uint8)
    out = Image.alpha_composite(base, Image.fromarray(arr, "RGBA"))
    if label_at is not None:                                                # the label sits on top of everything
        (x, y), col, txt = label_at; d2 = ImageDraw.Draw(out); f_ = get_font(max(11, int(13 * s)), True); w_ = d2.textlength(txt, font=f_)
        d2.rounded_rectangle([x - w_ / 2 - 6 * s, y - 12 * s, x + w_ / 2 + 6 * s, y + 10 * s], radius=int(5 * s), fill=col + (225,))
        d2.text((x, y - int(1 * s)), txt, font=f_, fill=(255, 255, 255, 255), anchor="mm")
    img[:] = cv2.cvtColor(np.array(out.convert("RGB")), cv2.COLOR_RGB2BGR)

def nearest_pairs(pf_f, def_team, att_team):
    """For each defending outfield player, the nearest outfield player of the team in possession (pf holds outfield players only)."""
    dfn, att = pf_f[pf_f.team == def_team], pf_f[pf_f.team == att_team]
    if len(dfn) == 0 or len(att) == 0:
        return []
    D = np.hypot(dfn.x.to_numpy()[:, None] - att.x.to_numpy()[None, :], dfn.y.to_numpy()[:, None] - att.y.to_numpy()[None, :])
    j = D.argmin(axis=1)
    out = []
    for i in range(len(dfn)):
        d_i = float(D[i, j[i]])
        if PAIR_MAX_DIST_M is None or d_i <= PAIR_MAX_DIST_M:
            out.append((int(dfn.track_id.iloc[i]), int(att.track_id.iloc[j[i]]), d_i,
                        float(dfn.x.iloc[i]), float(dfn.y.iloc[i]), float(att.x.iloc[j[i]]), float(att.y.iloc[j[i]])))
    return out

# ================================================================== minimap (absolute orientation: BAR always left -> right, ATM right -> left)
def draw_minimap(pos, ball_m, carrier_id, pair_xy, entry_x, entry_bgr):
    mm = np.zeros((MM_H, MM_W, 3), np.uint8); mm[:] = (40, 105, 40)
    sx, sy = MM_W / PITCH_LENGTH, MM_H / PITCH_WIDTH
    px = lambda x, y: (int(x * sx), int(y * sy))
    cv2.rectangle(mm, (0, 0), (MM_W - 1, MM_H - 1), (255, 255, 255), 2)
    cv2.line(mm, (MM_W // 2, 0), (MM_W // 2, MM_H), (255, 255, 255), 1)
    cv2.circle(mm, (MM_W // 2, MM_H // 2), int(9.15 * sx), (255, 255, 255), 1, cv2.LINE_AA)
    for gx in (0, PITCH_LENGTH - 16.5):
        cv2.rectangle(mm, (int(gx * sx), int((PITCH_WIDTH - 40.3) / 2 * sy)), (int((gx + 16.5) * sx), int((PITCH_WIDTH + 40.3) / 2 * sy)), (255, 255, 255), 1)
    for yy in range(0, MM_H, 12):                                             # entry line of the final third of the team in possession, dashed
        cv2.line(mm, (int(entry_x * sx), yy), (int(entry_x * sx), min(yy + 6, MM_H)), entry_bgr, 2)
    for (x1, y1, x2, y2, dist) in pair_xy:                                      # thin line defender -> his nearest opponent
        cv2.line(mm, px(x1, y1), px(x2, y2), pair_color(dist), 1, cv2.LINE_AA)
    for r in pos.itertuples():
        p = px(r.x, r.y); col = STYLE["team_bgr"][int(r.team)]
        cv2.circle(mm, p, 5, col, -1, cv2.LINE_AA)
        if r.role == "goalkeeper":
            cv2.circle(mm, p, 7, (255, 255, 255), 1, cv2.LINE_AA)
        num = NUM_OF.get(int(r.track_id))
        if num is not None:
            cv2.putText(mm, str(num), (p[0] + 7, p[1] + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.36, (255, 255, 255), 1, cv2.LINE_AA)
        if pd.notna(carrier_id) and r.track_id == carrier_id:
            cv2.fillPoly(mm, [np.array([[p[0] - 5, p[1] - 15], [p[0] + 5, p[1] - 15], [p[0], p[1] - 8]], np.int32)], (255, 255, 255))
    if ball_m is not None and not np.isnan(ball_m[0]):
        cv2.circle(mm, px(*ball_m), 3, STYLE["ball_bgr"], -1, cv2.LINE_AA)
    right = [t for t in (0, 1) if TEAM_ATTACKS_POSITIVE_X[t]][0]; left = 1 - right
    cv2.putText(mm, f"{TEAM_TXT[right]} attacks >", (6, MM_H - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
    t2 = f"< {TEAM_TXT[left]} attacks"; (tw, _), _ = cv2.getTextSize(t2, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
    cv2.putText(mm, t2, (MM_W - tw - 6, MM_H - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
    return mm

# ================================================================== HUD layer (header, status, lane legend, timeline, chips), drawn with a proper font
def draw_hud(img, ctx):
    H_, W_ = img.shape[:2]; s = H_ / 1080
    base = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).convert("RGBA")
    ov = Image.new("RGBA", base.size, (0, 0, 0, 0)); dr = ImageDraw.Draw(ov)
    f_title, f_sub = get_font(int(24 * s), True), get_font(int(14 * s), False)
    f_chip, f_row, f_rowb, f_small = get_font(int(18 * s), True), get_font(int(14 * s), False), get_font(int(14 * s), True), get_font(int(12 * s), False)

    # header: title + subtitle + team legend
    dr.rectangle([0, 0, W_, int(64 * s)], fill=(18, 18, 18, 170))
    dr.text((int(20 * s), int(22 * s)), ctx["title"], font=f_title, fill=(255, 255, 255, 255), anchor="lm")
    dr.text((int(20 * s), int(48 * s)), ctx["sub"], font=f_sub, fill=(205, 205, 205, 255), anchor="lm")
    lx = int(W_ * 0.52)
    for t in (0, 1):
        dr.rectangle([lx, int(16 * s), lx + int(16 * s), int(32 * s)], fill=TEAM_RGB[t] + (255,))
        dr.text((lx + int(24 * s), int(24 * s)), TEAM_TXT[t], font=f_rowb, fill=(255, 255, 255, 255), anchor="lm")
        lx += int(110 * s)

    # status chip (top right) + timer
    tw = dr.textlength(ctx["chip"], font=f_chip)
    cx1 = W_ - int(20 * s); cx0 = cx1 - int(tw + 28 * s)
    dr.rounded_rectangle([cx0, int(12 * s), cx1, int(52 * s)], radius=int(8 * s), fill=ctx["chip_rgb"] + (235,))
    dr.text(((cx0 + cx1) // 2, int(32 * s)), ctx["chip"], font=f_chip, fill=(255, 255, 255, 255), anchor="mm")
    if ctx.get("timer"):
        dr.text((cx0 - int(14 * s), int(32 * s)), ctx["timer"], font=f_sub, fill=(235, 235, 235, 255), anchor="rm")

    # pass-lane legend (counts of open / contested / blocked lanes from the carrier) + what the thin lines are
    if ctx.get("lanes") is not None:
        ln = ctx["lanes"]; x0, y0 = int(16 * s), int(76 * s)
        title_ = f"pass lanes from {ln['who']}"; w1 = dr.textlength(title_, font=f_rowb)
        dr.rounded_rectangle([x0, y0, x0 + int(max(w1 + 150 * s, 330 * s)), y0 + int(54 * s)], radius=int(7 * s), fill=(18, 18, 18, 150))
        dr.text((x0 + int(10 * s), y0 + int(16 * s)), title_, font=f_rowb, fill=(255, 255, 255, 255), anchor="lm")
        x = x0 + int(10 * s) + w1 + int(14 * s)
        for k in ("open", "contested", "blocked"):
            dr.ellipse([x, y0 + int(11 * s), x + int(10 * s), y0 + int(21 * s)], fill=LANE_RGB[k] + (255,))
            dr.text((x + int(15 * s), y0 + int(16 * s)), str(ln[k]), font=f_row, fill=(255, 255, 255, 255), anchor="lm"); x += int(34 * s)
        dr.text((x0 + int(10 * s), y0 + int(38 * s)), ctx["pairs_txt"], font=f_small, fill=(210, 210, 210, 255), anchor="lm")

    # distance chips on the pitch
    f_tiny = get_font(max(9, int(11 * s)), False)
    for x, y, text, rgb in ctx["chips"]:
        w_ = dr.textlength(text, font=f_tiny)
        dr.rounded_rectangle([x - w_ / 2 - 4 * s, y - 7 * s, x + w_ / 2 + 4 * s, y + 7 * s], radius=int(5 * s), fill=rgb + (215,))
        dr.text((x, y), text, font=f_tiny, fill=(255, 255, 255, 255), anchor="mm")

    # timeline (thin)
    tl = ctx["tl"]; frames, up = tl["frames"], tl["up"]; n = len(frames)
    x0, x1, y0, h = int(W_ * 0.12), int(W_ * 0.58), H_ - int(34 * s), int(10 * s)
    xs = lambda i: x0 + int((x1 - x0) * i / max(n - 1, 1))
    dr.rectangle([x0, y0, x1, y0 + h], fill=(70, 70, 70, 200))
    for a, b in bool_runs(up, frames):
        dr.rectangle([xs(a), y0, xs(b), y0 + h], fill=bgr2rgb(COL["hot"]) + (255,))
    for mark, rgb in ((tl["start"], bgr2rgb(COL["good"])), (tl["end"], tl["end_rgb"])):
        i = int(np.clip(mark - frames[0], 0, n - 1)); dr.rectangle([xs(i) - 1, y0 - int(5 * s), xs(i) + 1, y0 + h + int(5 * s)], fill=rgb + (255,))
    i = int(np.clip(tl["now"] - frames[0], 0, n - 1)); dr.rectangle([xs(i) - 2, y0 - int(8 * s), xs(i) + 2, y0 + h + int(8 * s)], fill=(255, 255, 255, 255))
    dr.text((x0 - int(10 * s), y0 + h // 2), "pressure", font=f_small, fill=(235, 235, 235, 255), anchor="rm")

    # outcome banner after the end of the build-up
    if ctx.get("outcome_txt"):
        f_big = get_font(int(30 * s), True); w_ = dr.textlength(ctx["outcome_txt"], font=f_big)
        dr.rounded_rectangle([W_ // 2 - w_ / 2 - 24 * s, int(84 * s), W_ // 2 + w_ / 2 + 24 * s, int(138 * s)], radius=int(10 * s), fill=ctx["outcome_rgb"] + (235,))
        dr.text((W_ // 2, int(111 * s)), ctx["outcome_txt"], font=f_big, fill=(255, 255, 255, 255), anchor="mm")

    img[:] = cv2.cvtColor(np.array(Image.alpha_composite(base, ov).convert("RGB")), cv2.COLOR_RGB2BGR)

# ================================================================== render one clip
def render_clip(bid, category, title):
    b = buildups_p[buildups_p.buildup_id == bid].iloc[0]
    team = int(b.team)
    anchor = int(b.first_press_frame) if pd.notna(b.first_press_frame) else int(b.start_frame)
    f1 = int(b.end_frame) + F(POST_SEC)
    f0 = max(0, anchor - F(PRE_SEC), f1 - F(MAX_CLIP_SEC))
    frames = np.arange(f0, f1 + 1)

    try:
        calib = pd.read_parquet(PLAYER_PATH, columns=CALIB_COLS, filters=[("frame_idx", ">=", f0), ("frame_idx", "<=", f1)])
    except Exception:
        calib = pd.read_parquet(PLAYER_PATH, columns=CALIB_COLS); calib = calib[(calib.frame_idx >= f0) & (calib.frame_idx <= f1)]
    if ALLOWED_PLAYER_SOURCES is not None:
        calib = calib[calib.source.isin(ALLOWED_PLAYER_SOURCES)]
    Hraw = fit_homographies(calib)                                       # per-frame camera mapping from the raw (unsmoothed) feet / pitch pairs
    Hs = smooth_homographies(Hraw, frames)
    boxes = smooth_boxes(calib)                                          # tags stay put: no jitter
    px_by_f = {k: g for k, g in boxes.groupby("frame_idx")}
    raw_by_f = {k: g for k, g in calib.groupby("frame_idx")}
    bpx = pd.read_parquet(BALL_PATH, columns=["frame_idx", "ball_px_x", "ball_px_y", "ball_source", "carrier_track_id"])
    bpx = bpx[(bpx.frame_idx >= f0) & (bpx.frame_idx <= f1)].set_index("frame_idx")
    pos_clip = player[(player.frame_idx >= f0) & (player.frame_idx <= f1)]
    pos_by_f = {k: g for k, g in pos_clip.groupby("frame_idx")}
    pf_clip = pf[(pf.frame_idx >= f0) & (pf.frame_idx <= f1)]               # smoothed outfield players (no goalkeepers)
    pf_by_f = {k: g for k, g in pf_clip.groupby("frame_idx")}
    ball_m = ball.set_index("frame_idx")[["ball_x", "ball_y"]]
    pr_win = pr_i.reindex(frames)
    up = pr_win["under_pressure"].fillna(False).to_numpy(bool)
    nprs = pr_win["n_pressing"].fillna(0).to_numpy(int)
    in_b = (frames >= b.start_frame) & (frames <= b.end_frame)
    cal_share = np.mean([Hs.get(int(f)) is not None for f in frames[in_b]]) * 100 if in_b.any() else 0.0
    cal_err = np.median([Hraw[int(f)]["err"] for f in frames[in_b] if int(f) in Hraw]) if any(int(f) in Hraw for f in frames[in_b]) else float("nan")
    print(f"    camera fit: {cal_share:.0f}% of the build-up frames, median reprojection error {cal_err:.2f} px")

    cap = cv2.VideoCapture(str(VIDEO_PATH)); cap.set(cv2.CAP_PROP_POS_FRAMES, f0 + FRAME_OFFSET)
    ok_, img0 = cap.read(); assert ok_, "cannot read the video at the clip start"
    H_, W_ = img0.shape[:2]; sc = H_ / 1080
    (CLIP_DIR / category).mkdir(parents=True, exist_ok=True)               # clips/lost_under_pressure, clips/kept_under_pressure, clips/lost_without_pressure
    out_path = CLIP_DIR / category / f"b{bid:03d}_{TEAM_TXT[team]}_{b.start_time.replace(':', '-')}.mp4"   # every clip is rendered again and overwritten
    vw = None
    for tag in (CODEC, "mp4v"):
        vw = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*tag), FPS, (W_, H_))
        if vw.isOpened():
            break
    outcome_bgr = {"success": COL["good"], "lost": COL["bad"]}.get(b.outcome, COL["neutral"])
    outcome_txt = {"success": "ENTERED THE FINAL THIRD", "lost": "BALL LOST"}.get(b.outcome, "SEQUENCE ENDED")
    right = [t for t in (0, 1) if TEAM_ATTACKS_POSITIVE_X[t]][0]
    mh, mw = int(MM_H * sc), int(MM_W * sc)

    for i, f in enumerate(frames):
        f = int(f)
        if i > 0:
            ok_, img0 = cap.read()
            if not ok_:
                break
        img = img0.copy()
        brow = bpx.loc[f] if f in bpx.index else None
        carrier_id = int(brow.carrier_track_id) if brow is not None and pd.notna(brow.carrier_track_id) else None
        in_build = b.start_frame <= f <= b.end_frame
        pressed_now = bool(up[i]) and in_build
        px_f = px_by_f.get(f)
        ball_xy = (brow.ball_px_x, brow.ball_px_y) if (brow is not None and pd.notna(brow.ball_px_x) and
                                                       (ALLOWED_BALL_SOURCES is None or brow.ball_source in ALLOWED_BALL_SOURCES)) else None
        pairs = nearest_pairs(pf_by_f[f], 1 - team, team) if (in_build and f in pf_by_f) else []
        lanes, chips, lane_info = [], [], None

        if in_build and px_f is not None:                                  # 1) ground graphics, behind the players
            foot = {int(r.track_id): (r.foot_px_x, r.foot_px_y) for r in px_f.itertuples() if pd.notna(r.foot_px_x)}
            pos_f = pos_by_f.get(f)
            lanes = compute_lanes(carrier_id, pos_f, foot) if (pos_f is not None and LAYERS["pass_lanes"]) else []
            pairs_px = [(foot[d_], foot[a_], dist) for d_, a_, dist, *_ in pairs if d_ in foot and a_ in foot]
            tgt = None
            if carrier_id is not None and f in raw_by_f:
                cr = raw_by_f[f][raw_by_f[f].track_id == carrier_id]
                if len(cr) and pd.notna(cr.iloc[0].pitch_x_raw):
                    tgt = (float(cr.iloc[0].pitch_x_raw), float(cr.iloc[0].pitch_y_raw))
            if tgt is None and f in loc_i.index and pd.notna(loc_i.loc[f, "x"]):
                tgt = (float(loc_i.loc[f, "x"]), float(loc_i.loc[f, "y"]))
            line_x = None
            if f in pf_by_f:
                deff = pf_by_f[f][pf_by_f[f].team == 1 - team]
                if len(deff):
                    line_x = float(deff.x.min()) if TEAM_ATTACKS_POSITIVE_X[1 - team] else float(deff.x.max())
            g = dict(team=team, target_m=tgt, n_press=int(nprs[i]), def_team=1 - team, def_line_x=line_x, lanes=lanes, pairs_px=pairs_px, layers=LAYERS)
            ground_layer(img, Hs.get(f), g, px_f)
            chips = [((p1[0] + p2[0]) // 2, (p1[1] + p2[1]) // 2, f"{dist:.1f}", bgr2rgb(pair_color(dist)))
                     for p1, p2, dist in pairs_px if dist <= PAIR_LABEL_MAX_M]
            if LAYERS["pass_lanes"] and carrier_id is not None:
                lane_info = dict(who=f"#{NUM_OF.get(carrier_id, '?')} {POS_OF.get(carrier_id, '')}",
                                 **{k: sum(1 for l in lanes if l[2] == k) for k in ("open", "contested", "blocked")})

        if px_f is not None:                                               # 2) tags, carrier marker and the small yellow ball
            draw_base_layer(img, px_f, ball_xy, carrier_id)
        elif ball_xy is not None:
            cv2.circle(img, (int(ball_xy[0]), int(ball_xy[1])), max(4, int(BALL_RADIUS_PX * sc)), STYLE["ball_bgr"], max(2, int(2 * sc)), cv2.LINE_AA)

        pos_f2 = pos_by_f.get(f)                                           # 3) minimap (bottom right, semi-transparent)
        if pos_f2 is not None:
            bm = tuple(ball_m.loc[f]) if f in ball_m.index else None
            mm = cv2.resize(draw_minimap(pos_f2, bm, carrier_id, [(p[3], p[4], p[5], p[6], p[2]) for p in pairs],
                                         70 if TEAM_ATTACKS_POSITIVE_X[team] else 35, STYLE["team_bgr"][team]), (mw, mh))
            x0_, y0_ = W_ - mw - int(20 * sc), H_ - mh - int(56 * sc)
            roi = img[y0_:y0_ + mh, x0_:x0_ + mw]; img[y0_:y0_ + mh, x0_:x0_ + mw] = cv2.addWeighted(mm, 0.8, roi, 0.2, 0)

        if f < b.start_frame:
            chip, chip_bgr = "BEFORE THE BUILD-UP", COL["neutral"]
        elif f <= b.end_frame:
            chip, chip_bgr = (f"UNDER PRESSURE  x{int(nprs[i])}", COL["hot"]) if pressed_now else ("NO PRESSURE", (90, 140, 90))
        else:
            chip, chip_bgr = outcome_txt, outcome_bgr
        ctx = dict(title=title, chip=chip, chip_rgb=bgr2rgb(chip_bgr), chips=chips, lanes=lane_info,
                   pairs_txt=f"thin lines: each {TEAM_TXT[1 - team]} outfield player to his nearest opponent",
                   sub=f"build-up #{bid}  ·  {TEAM_TXT[team]} in possession  ·  {mmss(f)}  ·  {TEAM_TXT[right]} attacks left to right, {TEAM_TXT[1 - right]} right to left",
                   timer=f"{(f - b.start_frame) / FPS:4.1f} s" if in_build else "",
                   tl=dict(frames=frames, up=up & (frames >= b.start_frame) & (frames <= b.end_frame), start=b.start_frame, end=b.end_frame,
                           now=f, end_rgb=bgr2rgb(outcome_bgr)),
                   outcome_txt=outcome_txt if f > b.end_frame else None, outcome_rgb=bgr2rgb(outcome_bgr))
        draw_hud(img, ctx)                                                 # 4) header, legend, timeline, outcome

        for _ in range(SLOWDOWN):
            vw.write(img)
    for _ in range(F(FREEZE_END_SEC) * SLOWDOWN):                              # hold the last frame
        vw.write(img)
    vw.release(); cap.release()
    return out_path

# ================================================================== render (skips clips that already exist)
import time
OVERWRITE = False          # True = re-render every clip, False = only build the missing ones

def clip_path(cat, bid):
    b = buildups_p[buildups_p.buildup_id == bid].iloc[0]
    return CLIP_DIR / cat / f"b{bid:03d}_{TEAM_TXT[int(b.team)]}_{b.start_time.replace(':', '-')}.mp4"

t_start = time.time(); done = []
for k, (cat, title, bid) in enumerate(selected, 1):
    p = clip_path(cat, bid)
    if p.exists() and p.stat().st_size > 0 and not OVERWRITE:
        done.append(p); print(f"[{k}/{len(selected)}] skipped (already exists) {cat}/{p.name}"); continue
    t0 = time.time()
    path = render_clip(bid, cat, title)
    done.append(path)
    print(f"[{k}/{len(selected)}] {cat}/{path.name}  ({path.stat().st_size / 1e6:.1f} MB, {time.time() - t0:.0f}s, total {(time.time() - t_start) / 60:.1f} min)")
print(f"{len(done)} clips in", CLIP_DIR)
for cat in categories:
    print(f"  {cat}: {len(list((CLIP_DIR / cat).glob('*.mp4')))} clips  ->  {CLIP_DIR / cat}")


### 14b. Check clips for build-ups that ended as dead / untracked
Renders `dead_or_untracked` build-ups into `buildup_output/dead_new/` so you can verify each one by eye. Fill the `verdict` column of `check_overview.csv` as you go.

In [ ]:
# ================================================================== CHECK CLIPS for build-ups that ended as dead_or_untracked
# Run AFTER the final clip cell (section 14): it uses render_clip, CLIP_DIR, buildups_p and the settings defined there.
# Purpose: watch them and decide per case whether the label is right (ball out / foul / whistle),
# or the tracking lost the ball / carrier (then fix it in REVIEW and re-run from section 2b).
CHECK_END_DETAILS = None          # None = all, or a list e.g. ["out_or_stoppage_then_opponent", "same_team_after_gap"]
CHECK_TEAM = None                 # None = both teams, or "ATM" / "BAR"
MAX_CHECK_CLIPS = None            # None = all, or a number (longest first is NOT applied: they are rendered in match order)
OVERWRITE_CHECK = False
DEAD_DIR = OUT_DIR / "dead_new"   # new folder next to clips3/ (buildup_output/dead_new)

dead = buildups_p[buildups_p.outcome == "dead_or_untracked"].copy()
if CHECK_END_DETAILS is not None:
    dead = dead[dead.end_detail.isin(CHECK_END_DETAILS)]
if CHECK_TEAM is not None:
    dead = dead[dead.team_name == CHECK_TEAM]
dead = dead.sort_values("start_frame")
if MAX_CHECK_CLIPS is not None:
    dead = dead.head(MAX_CHECK_CLIPS)

print(f"{len(dead)} dead_or_untracked build-ups to check")
print(dead.groupby(["team_name", "end_detail"]).size().unstack(fill_value=0).to_string())
print()

def _dead_path(b):
    return DEAD_DIR / f"b{int(b.buildup_id):03d}_{TEAM_TXT[int(b.team)]}_{b.start_time.replace(':', '-')}.mp4"

t_start = time.time(); ok, failed = [], []
_clip_dir_saved = CLIP_DIR
CLIP_DIR = OUT_DIR                # render_clip writes to CLIP_DIR / category, so category "dead" -> OUT_DIR / "dead"
for k, b in enumerate(dead.itertuples(), 1):
    p = _dead_path(b)
    if p.exists() and p.stat().st_size > 0 and not OVERWRITE_CHECK:
        ok.append(p); print(f"[{k}/{len(dead)}] skipped (exists) {p.name}"); continue
    t0 = time.time()
    try:
        path = render_clip(int(b.buildup_id), "dead", f"CHECK: ended as {b.end_detail}")
        ok.append(path)
        print(f"[{k}/{len(dead)}] {path.name}  {b.team_name} {b.start_time} -> ends {mmss(b.end_frame)}  ({b.end_detail}; {path.stat().st_size / 1e6:.1f} MB, {time.time() - t0:.0f}s, total {(time.time() - t_start) / 60:.1f} min)")
    except Exception as e:
        failed.append((int(b.buildup_id), repr(e))); print(f"[{k}/{len(dead)}] build-up {int(b.buildup_id)} FAILED: {e!r}")

CLIP_DIR = _clip_dir_saved        # restore, so the other clip cells keep writing to clips3/
print(f"\n{len(ok)} clips in {DEAD_DIR}")
if failed:
    print("failed:", failed)

# overview table to keep next to the clips: write your verdict in the last column of the CSV
overview = dead[["buildup_id", "team_name", "start_time", "duration_sec", "end_detail", "end_x_att", "end_y_att"]].copy()
overview["end_time"] = dead["end_frame"].map(mmss)
overview["verdict"] = ""          # e.g. ball_out / foul / fake_carrier / tracking_lost / ok
DEAD_DIR.mkdir(parents=True, exist_ok=True)
overview_path = DEAD_DIR / "check_overview.csv"
overview.to_csv(overview_path, index=False)
print("overview saved to", overview_path)


## 15. LLM prompt for one clip
Put a clip name in `CLIP_NAMES`. The cell prints a ready-to-paste prompt asking an LLM why the pressure succeeded or failed, and saves it to `prompts/`. It needs neither the video nor the clips.

In [ ]:
# ================================================================== LLM PROMPT for one clip (why did the pressure succeed / fail?)
# needs (all already in memory after the analysis cells): buildups_p, pr_i, loc_i, pf, d, ball, roster, POS_OF, NUM_OF,
#   TEAM_NAMES, TEAM_ATTACKS_POSITIVE_X, P, F, FPS, bool_runs, mmss, third_of, PITCH_LENGTH, PITCH_WIDTH, OUT_DIR
# usage: put one or more clip names (file name or path, e.g. "b134_ATM_36-24.mp4") in CLIP_NAMES and run the cell.
#        The prompt is printed (copy it into the LLM) and also saved to  OUT_DIR / "prompts" / <clip>.txt
import re

CLIP_NAMES = ["b134_ATM_36-24.mp4"]          # <- the clip(s) you want to analyse
SNAP_FREE_MIN_M = 4.0                          # a teammate is a "free option" if his nearest opponent is farther than this
SNAP_OPTION_MAX_M = 35.0                       # ...and he is within this distance of the ball
TOP_PRESSERS = 4                               # pressers listed per pressure episode

_needed = ["buildups_p", "pr_i", "loc_i", "pf", "d", "ball", "roster", "POS_OF", "NUM_OF", "TEAM_NAMES",
           "TEAM_ATTACKS_POSITIVE_X", "P", "F", "FPS", "bool_runs", "mmss", "third_of", "PITCH_LENGTH", "PITCH_WIDTH", "OUT_DIR"]
_missing = [n for n in _needed if n not in globals()]
assert not _missing, f"run the earlier cells first, these are missing: {_missing}"

PROMPT_DIR = OUT_DIR / "prompts"; PROMPT_DIR.mkdir(exist_ok=True)
NAME_OF = roster.set_index("track_id")["name"].to_dict()


def who(tid):
    tid = int(tid)
    return f"#{NUM_OF.get(tid)} {NAME_OF.get(tid, '?')} ({POS_OF.get(tid, '?')})"


def to_att(x, y, team):
    """raw pitch metres -> the attacking frame of `team` (x = metres from own goal)."""
    return (x, y) if TEAM_ATTACKS_POSITIVE_X[int(team)] else (PITCH_LENGTH - x, PITCH_WIDTH - y)


def fmt(v, nd=1):
    return "n/a" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{nd}f}"


def snapshot(fr, team, pfw):
    """Situation of the ball carrier at one frame: pressure around him and his passing options (all from tracking data)."""
    if fr not in loc_i.index:
        return None
    tx, ty = loc_i.at[fr, "x"], loc_i.at[fr, "y"]
    if pd.isna(tx):
        return None
    cid = loc_i.at[fr, "carrier_track_id"]
    g = pfw[pfw.frame_idx == fr]
    mates = g[(g.team == team) & (g.track_id != (cid if pd.notna(cid) else -1))]
    opp = g[g.team != team]
    if opp.empty:
        return None
    ax = lambda x: x if TEAM_ATTACKS_POSITIVE_X[int(team)] else PITCH_LENGTH - x
    t_ax = ax(tx)
    od = np.hypot(opp.x - tx, opp.y - ty)
    out = dict(
        frame=fr, carrier=who(cid) if pd.notna(cid) else "no carrier (ball only)",
        x_att=float(t_ax), third=third_of(t_ax),
        opp_within_5=int((od < 5).sum()), opp_within_10=int((od < 10).sum()),
        opp_goalside=int((ax(opp.x) > t_ax).sum()),               # opponents between the ball and the goal being attacked
        opp_block_len=float(ax(opp.x).max() - ax(opp.x).min()), opp_block_wid=float(opp.y.max() - opp.y.min()),
    )
    md = np.hypot(mates.x - tx, mates.y - ty)
    out["mates_within_10"] = int((md < 10).sum())
    opts = []
    for m, dm in zip(mates.itertuples(), md):
        if dm > SNAP_OPTION_MAX_M:
            continue
        nearest_opp = float(np.hypot(opp.x - m.x, opp.y - m.y).min())
        opts.append(dict(who=who(m.track_id), dist=float(dm), free=nearest_opp > SNAP_FREE_MIN_M,
                         forward=bool(ax(m.x) > t_ax + 1), opp_gap=nearest_opp))
    out["options"] = sorted(opts, key=lambda o: (not o["free"], not o["forward"], o["dist"]))
    out["n_free"] = sum(o["free"] for o in opts); out["n_free_fwd"] = sum(o["free"] and o["forward"] for o in opts)
    return out


def snap_text(s, label):
    if s is None:
        return f"{label}: no usable snapshot (tracking gap)."
    lines = [f"{label} (t={mmss(s['frame'])}): ball/carrier {s['carrier']}, {fmt(s['x_att'])} m from own goal ({s['third']}).",
             f"  opponents within 5 m: {s['opp_within_5']}, within 10 m: {s['opp_within_10']}, goal-side of the ball: {s['opp_goalside']}; "
             f"their block is {fmt(s['opp_block_len'])} m long x {fmt(s['opp_block_wid'])} m wide.",
             f"  teammates within 10 m: {s['mates_within_10']}; free options (nearest opponent > {SNAP_FREE_MIN_M:.0f} m, within "
             f"{SNAP_OPTION_MAX_M:.0f} m): {s['n_free']}, of which ahead of the ball: {s['n_free_fwd']}."]
    for o in s["options"][:4]:
        lines.append(f"    - {o['who']}: {o['dist']:.0f} m from the ball, {'FREE' if o['free'] else 'marked'} "
                     f"(nearest opponent {o['opp_gap']:.1f} m), {'ahead' if o['forward'] else 'level/behind'}")
    return "\n".join(lines)


def build_prompt(clip_name):
    p = Path(clip_name)
    m = re.search(r"b(\d+)_", p.name)
    if not m:
        raise ValueError(f"cannot read the build-up id from '{clip_name}' (expected e.g. b134_ATM_36-24.mp4)")
    bid = int(m.group(1))
    rows = buildups_p[buildups_p.buildup_id == bid]
    if rows.empty:
        raise KeyError(f"build-up {bid} not found in buildups_p")
    b = rows.iloc[0]
    team = int(b.team); opp_team = 1 - team
    tn, on = TEAM_NAMES[team], TEAM_NAMES[opp_team]
    a, z = int(b.start_frame), int(b.end_frame)
    category = p.parent.name if p.parent.name in ("lost_under_pressure", "kept_under_pressure", "lost_without_pressure") else "not given"

    seg = pr_i.loc[a:z]
    pfw = pf[(pf.frame_idx >= a - 1) & (pf.frame_idx <= z + 1)]
    dw = d[(d.frame_idx >= a) & (d.frame_idx <= z)]

    # ---- result labels (build-up team's view and pressing team's view)
    pressed = bool(b.get("pressed_any", False))
    if b.outcome == "lost":
        press_view = f"PRESS SUCCEEDED: {on} won the ball back" + ("" if pressed else " (but no sustained pressure was measured before the loss)")
    elif b.outcome == "success":
        press_view = f"PRESS FAILED: {tn} progressed into the attacking third" + (" despite being pressed" if pressed else " (it was never sustainedly pressed)")
    else:
        press_view = "INCONCLUSIVE: the build-up ended in a stoppage / untracked phase, neither a clear win nor a clear escape"

    # ---- pressure episodes
    ep_txt = []
    if len(seg):
        up = seg["under_pressure"].to_numpy(bool); fr_ = seg.index.to_numpy()
        for k, (s, e) in enumerate(bool_runs(up, fr_), 1):
            fa, fb = int(fr_[s]), int(fr_[e])
            sub = seg.iloc[s:e + 1]
            xa_s = loc_i.at[fa, "x_att"] if fa in loc_i.index else np.nan
            xa_e = loc_i.at[fb, "x_att"] if fb in loc_i.index else np.nan
            ep = dw[(dw.frame_idx >= fa) & (dw.frame_idx <= fb) & dw.is_press]
            top = (ep.groupby("track_id").agg(frames=("frame_idx", "size"), min_dist=("dist", "min"), closing=("closing", "mean"))
                   .sort_values("frames", ascending=False).head(TOP_PRESSERS))
            pres = "; ".join(f"{who(t)} pressing {r.frames / FPS:.1f}s, closest {r.min_dist:.1f} m, mean closing speed {fmt(r.closing)} m/s"
                             for t, r in top.iterrows()) or "no individual presser identified"
            ep_txt.append(f"  Episode {k}: {mmss(fa)} -> {mmss(fb)} ({(fb - fa + 1) / FPS:.1f}s, {(fa - a) / FPS:.1f}s after the build-up started); "
                          f"ball moved from {fmt(xa_s)} m to {fmt(xa_e)} m (own-goal frame); max pressers {int(sub.n_pressing.max())}, "
                          f"closest defender {sub.min_dist.min():.1f} m.\n    Pressers: {pres}")
    ep_block = "\n".join(ep_txt) if ep_txt else "  none (no defender pressed for the minimum sustained time)"

    # ---- second-by-second timeline
    tl = []
    step = max(1, int(round(FPS)))
    for t in list(range(a, z + 1, step)) + ([z] if (z - a) % step else []):
        if t not in pr_i.index:
            tl.append(f"  {(t - a) / FPS:5.1f}s  {mmss(t)}  (no press target: ball lost to tracking / dead ball)"); continue
        r = pr_i.loc[t]
        cid = loc_i.at[t, "carrier_track_id"] if t in loc_i.index else np.nan
        xa_t = loc_i.at[t, "x_att"] if t in loc_i.index else np.nan
        ya_t = loc_i.at[t, "y_att"] if t in loc_i.index else np.nan
        tl.append(f"  {(t - a) / FPS:5.1f}s  {mmss(t)}  x={fmt(xa_t):>5} y={fmt(ya_t):>5}  carrier={who(cid) if pd.notna(cid) else '-':<28} "
                  f"pressers={int(r.n_pressing)} near7m={int(r.n_near)} closest={r.min_dist:4.1f} m  {'UNDER PRESSURE' if r.under_pressure else ''}")
    tl_block = "\n".join(tl)

    # ---- situation at key moments
    t_first = int(b.first_press_frame) if pd.notna(b.get("first_press_frame", np.nan)) else None
    t_dec = max(a, z - int(FPS))                       # 1 s before the end: the decision / loss moment
    snaps = []
    if t_first is not None:
        snaps.append(snap_text(snapshot(t_first, team, pfw), "At the FIRST pressure"))
    snaps.append(snap_text(snapshot(t_dec, team, pfw), "1 s BEFORE THE END"))
    snap_block = "\n".join(snaps)

    # ---- how it ended
    end_lines = [f"Outcome: {b.outcome} ({b.end_detail}). Last known ball position: {fmt(b.end_x_att)} m from own goal, "
                 f"y={fmt(b.end_y_att)} (own-goal frame, {third_of(b.end_x_att) if pd.notna(b.end_x_att) else 'n/a'})."]
    last = dw[(dw.frame_idx > z - int(FPS)) & dw.is_press]
    if len(last):
        lp = last.groupby("track_id").size().sort_values(ascending=False).head(3)
        end_lines.append("Pressing in the last second: " + "; ".join(f"{who(t)} ({n / FPS:.1f}s)" for t, n in lp.items()) + ".")
    else:
        end_lines.append("No defender was pressing in the last second.")
    if b.outcome == "success":
        end_lines.append(f"The ball crossed the entry line and held there {P['ENTRY_HOLD_SEC']}s; deepest point {fmt(b.max_x_att - P['THIRD_2_M'])} m past the line.")
    else:
        nxt = ball[(ball.frame_idx > z) & ball.carrier_track_id.notna()].head(1)
        if len(nxt):
            n0 = nxt.iloc[0]; fN = int(n0.frame_idx)
            gap_stop = bool(ball[(ball.frame_idx > z) & (ball.frame_idx <= fN)].is_stoppage.any())
            nx = to_att(n0.ball_x, n0.ball_y, team) if pd.notna(n0.ball_x) else (np.nan, np.nan)
            end_lines.append(f"Next player in control: {who(n0.carrier_track_id)} of {TEAM_NAMES.get(int(n0.carrier_team), '?')}, "
                             f"{(fN - z) / FPS:.1f}s after the build-up ended, ball at {fmt(nx[0])} m (pressed team's own-goal frame)"
                             f"{'; the ball went through a stoppage / out of play in between' if gap_stop else ''}.")
    end_block = "\n".join(end_lines)

    # ---- data quality
    src = seg["src"].value_counts(normalize=True).mul(100).round(0).astype(int).to_dict() if len(seg) else {}
    dq = (f"press target coverage {b.get('target_coverage', np.nan) * 100:.0f}% of the build-up's frames; press-target source (% of frames) {src}; "
          f"defenders tracked per frame: median {int(seg.n_def_tracked.median()) if len(seg) else 'n/a'}, minimum {int(seg.n_def_tracked.min()) if len(seg) else 'n/a'} (of 10 outfield).")

    prompt = f"""You are a football tactical analyst. Below is tracking-derived data for ONE build-up phase from {tn} vs {on} (La Liga, first half). Your job is to explain WHY the pressure by {on} succeeded or failed in this phase.

RULES
- Use only the data below. Do not invent events, players or passes that are not supported by it. If something cannot be known from the data (for example the type of pass, body orientation, the referee's decision), say so explicitly.
- Every claim must be backed by a number or an observation from the data (distance, count of pressers, timing, position).
- If a video of the clip is attached, mark every statement that comes from the video as [VIDEO] so it can be separated from the tracking data.

DEFINITIONS (how the numbers were produced)
- Build-up = a possession phase that starts in the team's own or middle third and ends when the ball holds beyond {P['THIRD_2_M']:.0f} m from its own goal (success), is lost to the opponent (lost), or ends in a stoppage / untracked phase.
- Positions are in metres in the build-up team's attacking frame: x = distance from its own goal (0 = own goal line, {PITCH_LENGTH:.0f} = opponent goal line), y = width (0 to {PITCH_WIDTH:.0f}). Thirds: own < {P['THIRD_1_M']:.0f} m, middle {P['THIRD_1_M']:.0f} to {P['THIRD_2_M']:.0f} m, attacking > {P['THIRD_2_M']:.0f} m.
- Pressing defender = within {P['PRESS_DIST_M']:.0f} m of the ball carrier and either within {P['PRESS_TIGHT_M']:.0f} m, or closing in at >= {P['CLOSING_SPEED_MPS']} m/s (measured over {P['CLOSING_WINDOW_SEC']}s).
- "Under pressure" = at least one pressing defender for >= {P['PRESS_SUSTAIN_SEC']}s in a row.
- Free option = a teammate whose nearest opponent is more than {SNAP_FREE_MIN_M:.0f} m away. This is a geometric proxy, not a guarantee that a pass was possible (passing lanes and body orientation are not modelled).
- Positions (CB, CM, ...) are INFERRED from average location, so treat labels as approximate.

CLIP
- File: {p.name}   |   build-up id {bid}   |   category: {category}
- Build-up team: {tn}   |   pressing team: {on}
- Time: {b.start_time} to {mmss(z)} ({b.duration_sec:.1f}s)   |   started in the {b.start_third} ({b.start_reason}) at x={fmt(b.start_x_att)} m
- Result: build-up {b.outcome.upper()}  ->  {press_view}
- Pressure summary: pressed at some point = {pressed}; share of frames under pressure = {fmt(b.get('pressure_share', np.nan) * 100, 0)}%; pressure episodes = {b.get('n_pressure_episodes', 0)}; max pressers = {b.get('max_pressers', 'n/a')}; time to first pressure = {fmt(b.get('time_to_first_pressure_sec', np.nan))}s; pressed from the start (counter-press) = {b.get('pressed_from_start', False)}; first pressure in the {b.get('first_press_third', 'n/a')}, channel {b.get('first_press_channel', 'n/a')}; pressed at the end = {b.get('pressed_at_end', 'n/a')}.

PRESSURE EPISODES
{ep_block}

SITUATION AT KEY MOMENTS
{snap_block}

SECOND-BY-SECOND TIMELINE
{tl_block}

HOW IT ENDED
{end_block}

DATA QUALITY
- {dq}
- The pressure measure is geometric (distance and closing speed to the ball carrier or the ball). It does not see passes, body shape or pressing triggers.

WHAT I WANT FROM YOU
1. VERDICT (2 sentences): did the pressing team's pressure work or fail in this clip, and was the result really caused by pressure or by something else (own error, long ball, loose touch, stoppage, tracking issue)?
2. WHAT HAPPENED: a short chronological reading of the phase (4 to 6 bullets) using the timeline.
3. WHY: the 2 to 4 main reasons, ranked, each with its evidence. Cover the pressing side (who pressed, how fast they closed, how many, the angle and the cover behind) and the build-up side (how many free options, how far, whether the carrier was isolated, how the block behind the ball was set).
4. COUNTERFACTUAL: one thing the losing side could have done differently, based only on the data above.
5. CONFIDENCE: low / medium / high, and which parts of the data limit that confidence.
Keep it under 350 words, plain language, no filler."""
    return prompt


for name in CLIP_NAMES:
    txt = build_prompt(name)
    out = PROMPT_DIR / (Path(name).stem + ".txt")
    out.write_text(txt, encoding="utf-8")
    print("=" * 110); print(txt); print("=" * 110)
    print(f"saved to {out}\n")


## 16. Export for the pressure-stats notebook
Saves all tables and settings so the stats notebook runs standalone (no video, no re-detection). Run at the end.

In [ ]:
# ================================================================== EXPORT for the pressure-stats notebook (run at the END of this notebook)
import json
# Saves every table the stats notebook needs, plus the settings used, so the new notebook runs on its own (no video, no re-detection).
EXPORT_DIR = OUT_DIR / "pressure_stats_input"; EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def _clean(df):
    df = df.copy()
    for c in df.columns:
        if str(df[c].dtype) == "category":
            df[c] = df[c].astype(str)
    return df

# 1) one row per build-up, with the pressure features from Cell 9
buildups_p_out = _clean(buildups_p)

# 2) one row per possession frame that has a press target: pressure signal + where the ball / carrier is
loc_cols = [c for c in ["frame_idx", "carrier_track_id", "x", "y", "x_att", "y_att", "third"] if c in loc.columns]
frames_out = _clean(pr.drop(columns=[c for c in ["pressed_team"] if c in pr.columns]).merge(loc[loc_cols], on="frame_idx", how="left"))

# 3) one row per defender and frame: distance to the pressed player, closing speed, pressing flag
d_cols = [c for c in ["frame_idx", "spell_id", "track_id", "press_team", "dist", "closing", "is_press", "is_near"] if c in d.columns]
defenders_out = _clean(d[d_cols])

# 4) player positions in possession frames (all outfield players + goalkeepers), for lanes / line height / compactness later
poss_frames = set(frames_out["frame_idx"].unique())
positions_out = _clean(player[player.frame_idx.isin(poss_frames)][["frame_idx", "track_id", "team", "role", "x", "y"]])

# 5) possession spells, roster with positions, settings
spells_out = _clean(spells)
roster_out = _clean(roster[["track_id", "team", "role", "name", "number", "position"]])
meta = dict(
    team_names={int(k): str(v) for k, v in TEAM_NAMES.items()},
    team_attacks_positive_x={str(k): bool(v) for k, v in TEAM_ATTACKS_POSITIVE_X.items()},
    fps=FPS, pitch_length=PITCH_LENGTH, pitch_width=PITCH_WIDTH,
    params={k: (v if isinstance(v, (int, float, str, bool)) else str(v)) for k, v in P.items()},
    n_buildups=int(len(buildups_p_out)), n_frames=int(len(frames_out)), minutes=float(frames_out["frame_idx"].max() / FPS / 60),
    note="first half only; teams: 0 and 1 as in team_names; positions are inferred, see roster.position",
)
for name, df in [("buildups", buildups_p_out), ("frames", frames_out), ("defenders", defenders_out),
                 ("positions", positions_out), ("spells", spells_out), ("roster", roster_out)]:
    df.to_parquet(EXPORT_DIR / f"{name}.parquet", index=False)
with open(EXPORT_DIR / "meta.json", "w") as f_:
    json.dump(meta, f_, indent=2)

print("saved to", EXPORT_DIR)
for p in sorted(EXPORT_DIR.iterdir()):
    print(f"  {p.name:20s} {p.stat().st_size / 1e6:7.2f} MB")
print()
for name, df in [("buildups", buildups_p_out), ("frames", frames_out), ("defenders", defenders_out), ("positions", positions_out), ("spells", spells_out)]:
    print(f"{name:10s} {len(df):>9,} rows x {df.shape[1]:>2} cols: {list(df.columns)[:12]}{' ...' if df.shape[1] > 12 else ''}")